# 02 — Data Understanding: Profil Kualitas 33 File Data Mentah
**DA Gelar Rasa 2026 · Dashboard Kualitas Udara & Kesehatan**

**Tujuan.** Mendokumentasikan kondisi data mentah di `data/raw/` *sebelum* dibersihkan: inventaris, kamus data, relasi antar-tabel,
kualitas per stasiun, statistik awal, dan profil data kesehatan.

**Aturan tahap ini.**
- Data mentah **tidak diubah** dan **tidak dibersihkan**. Semua file dibaca sebagai teks (`dtype=str`) agar teks galat dan string kosong tidak hilang.
- Konversi satuan (ppm → µg/m³) di notebook ini hanya dipakai untuk *membandingkan skala* dan menguji hipotesis, bukan untuk mengubah data.
- Seluruh fungsi ada di `src/profiling.py`; semua angka pada laporan/dashboard berikutnya harus berasal dari tabel keluaran notebook ini
  (`data/processed/profiling/`, `reports/data_quality_issues.csv`, `reports/data_dictionary.csv`).

| Bagian | Isi |
|---|---|
| 1 | Inventaris 33 file |
| 2 | Kamus data gabungan |
| 3 | Relasi stasiun–sensor–lokasi–RS |
| 4 | Profil kualitas per stasiun (waktu, nilai bermasalah, `qc_flag`) + visual kelengkapan |
| 5 | Statistik deskriptif awal & penandaan stasiun mencurigakan |
| 6 | Verifikasi hipotesis satuan ppm dan masalah yang sudah diketahui |
| 7 | Profil data kesehatan |
| 8 | Daftar masalah kualitas data (CSV) + ringkasan visual |
| 9 | Menyimpan tabel profil · 10. Temuan |

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display, Markdown

from src import config as cfg, style, profiling as P

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)
pio.renderers.default = "plotly_mimetype+notebook_connected"

cfg.PROC_DIR.mkdir(parents=True, exist_ok=True)
cfg.FIG_DIR.mkdir(parents=True, exist_ok=True)
cfg.FINDINGS_DIR.mkdir(parents=True, exist_ok=True)
PROF_DIR = cfg.PROC_DIR / "profiling"
PROF_DIR.mkdir(parents=True, exist_ok=True)

rp = lambda x: f"{int(round(x)):,}".replace(",", ".")      # pemisah ribuan gaya Indonesia
def tandai_positif(df, kolom):
    """Beri warna sel > 0 pada kolom hitungan masalah."""
    return df.style.map(lambda v: "background-color:#FDE9C8" if isinstance(v, (int, float, np.integer, np.floating)) and v > 0 else "", subset=kolom)
print("ROOT:", ROOT)
print(f"Periode AQMS: {cfg.WINDOW_START:%Y-%m-%d %H:%M} s.d. {cfg.WINDOW_END:%Y-%m-%d %H:%M} -> {rp(cfg.EXPECTED_HOURS)} jam diharapkan per stasiun")

ROOT: C:\Users\ACER\Downloads\BPS_Bismillah Pasti Satu
Periode AQMS: 2024-01-01 00:00 s.d. 2026-10-14 23:00 -> 24.432 jam diharapkan per stasiun


## 1. Inventaris 33 file
Kolom `tipe_data_terbaca` adalah tipe hasil inferensi pandas. Bila kolom yang seharusnya numerik terbaca sebagai `object`,
berarti ada teks di dalamnya (lihat `kolom_numerik_terbaca_teks`).

In [2]:
inv, detail = P.inventaris()
assert len(inv) == 33, f"Diharapkan 33 file, ditemukan {len(inv)}"

tampil = inv.drop(columns=["tabel"]).copy()
for c in ["waktu_awal", "waktu_akhir"]:
    tampil[c] = tampil[c].dt.strftime("%Y-%m-%d %H:%M").fillna("-")
display(tampil)

print("\nRingkasan per kelompok:")
display(inv.groupby("kelompok").agg(jumlah_file=("file", "size"), total_baris=("n_baris", "sum"), ukuran_mb=("ukuran_mb", "sum")).round(1))
print("Satuan polutan pada 27 file AQMS:", inv[inv.tabel == "aqms"].satuan.value_counts().to_dict())
print("File AQMS dengan kolom numerik terbaca sebagai teks:", inv[inv.kolom_numerik_terbaca_teks != ""].file.tolist())

,file,kelompok,kode_stasiun,n_baris,n_kolom,tipe_data_terbaca,kolom_numerik_terbaca_teks,waktu_awal,waktu_akhir,satuan,ukuran_mb
0,BGL-01_Bengkulu_AQMS.csv,Udara (AQMS),BGL-01,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:02,2026-10-14 23:02,ug/m3,3.09
1,BGR-01_Cisarua_AQMS.csv,Udara (AQMS),BGR-01,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:00,2026-10-14 23:00,ug/m3,2.95
2,BGR-02_Baranangsiang_AQMS.csv,Udara (AQMS),BGR-02,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:00,2026-10-14 23:00,ug/m3,2.95
3,BJB-01_Banjarbaru_AQMS.csv,Udara (AQMS),BJB-01,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:01,2026-10-14 23:01,ppm,3.40
4,BTM-01_Batam_AQMS.csv,Udara (AQMS),BTM-01,24432,15,"object×5, float64×9, int64×1",,2023-12-31 23:48,2026-10-14 22:48,ug/m3,3.10
5,HLT-01_health_outcomes_daily.csv,Kesehatan,None,27491,14,"object×3, float64×11",,2023-12-31 00:00,2026-10-14 00:00,jumlah kasus; °C; %,2.20
6,hospitals.csv,Tabel master,None,27,7,"object×6, float64×1",,-,-,-,0.00
7,IDP-01_Indrapuri_AQMS.csv,Udara (AQMS),IDP-01,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:00,2026-10-14 23:00,ug/m3,2.94
8,JKT-01_Gambir_AQMS.csv,Udara (AQMS),JKT-01,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:00,2026-10-14 23:00,ug/m3,2.98
9,JKT-02_Marunda_AQMS.csv,Udara (AQMS),JKT-02,24432,15,"object×5, float64×9, int64×1",,2024-01-01 00:00,2026-10-14 23:00,ug/m3,2.99



Ringkasan per kelompok:


,jumlah_file,total_baris,ukuran_mb
kelompok,,,
Kesehatan,1,27491,2.2
Metadata,2,43,0.0
Tabel master,3,81,0.0
Udara (AQMS),27,667981,85.2


Satuan polutan pada 27 file AQMS: {'ug/m3': 24, 'ppm': 3}
File AQMS dengan kolom numerik terbaca sebagai teks: ['JMB-02_MuaroJambi_AQMS.csv']


## 2. Kamus data gabungan
27 file AQMS digabung menjadi satu tabel logis. Kolom `*_ppm` hanya ada di 3 file. Tersimpan di `reports/data_dictionary.csv`.

In [3]:
kamus = P.kamus_data(detail)
kamus.to_csv(cfg.REPORTS_DIR / "data_dictionary.csv", index=False, encoding="utf-8-sig")
print(f"{len(kamus)} baris kamus -> reports/data_dictionary.csv")
display(kamus.groupby("tabel").size().rename("jumlah_kolom").to_frame())
display(kamus)

91 baris kamus -> reports/data_dictionary.csv


,jumlah_kolom
tabel,
AQMS (27 file),21
HLT-01_health_outcomes_daily.csv,14
hospitals.csv,7
metadata_dates.csv,6
metadata_sensors.csv,6
sensors.csv,10
sites.csv,27


,tabel,kolom,tipe_konseptual,tipe_pandas,satuan,deskripsi,contoh_nilai,jumlah_file,catatan
0,AQMS (27 file),measurement_id,teks (ID),object,-,ID unik baris pengukuran,MEAS-014-000001,27,
1,AQMS (27 file),sensor_id,teks (ID),object,-,ID sensor/stasiun yang merekam (relasi ke sensors.csv),SENS-014,27,
2,AQMS (27 file),timestamp_start,tanggal/waktu,object,-,Awal jendela pengukuran (WIB),2024-01-01 00:02:10.000,27,
3,AQMS (27 file),timestamp_end,tanggal/waktu,object,-,Akhir jendela pengukuran (WIB),2024-01-01 01:02:10.000,27,
4,AQMS (27 file),pm25_ug_m3,numerik (desimal),float64,µg/m³,"Partikel halus PM2.5 (diameter < 2,5 µm)",20.6,24,"hanya ada di 24 dari 27 file; tipe terbaca tidak seragam antar-file: float64×23, object×1"
5,AQMS (27 file),pm10_ug_m3,numerik (desimal),float64,µg/m³,Partikel PM10 (diameter < 10 µm),29.8,24,"hanya ada di 24 dari 27 file; tipe terbaca tidak seragam antar-file: float64×23, object×1"
6,AQMS (27 file),so2_ug_m3,numerik (desimal),float64,µg/m³,Sulfur dioksida (SO2),7.7,24,"hanya ada di 24 dari 27 file; tipe terbaca tidak seragam antar-file: float64×23, object×1"
7,AQMS (27 file),co_ug_m3,numerik (desimal),float64,µg/m³,Karbon monoksida (CO),363.4,24,"hanya ada di 24 dari 27 file; tipe terbaca tidak seragam antar-file: float64×23, object×1"
8,AQMS (27 file),o3_ug_m3,numerik (desimal),float64,µg/m³,Ozon permukaan (O3),11.4,24,"hanya ada di 24 dari 27 file; tipe terbaca tidak seragam antar-file: float64×23, object×1"
9,AQMS (27 file),no2_ug_m3,numerik (desimal),float64,µg/m³,Nitrogen dioksida (NO2),19.7,24,"hanya ada di 24 dari 27 file; tipe terbaca tidak seragam antar-file: float64×23, object×1"


## 3. Relasi stasiun–sensor–lokasi–RS
Rantai kunci: `AQMS.sensor_id → sensors.sensor_id → sensors.location_id → sites.location_id`, dan pasangan RS mengikuti nomor urut
(`SENS-0NN ↔ RS-XXX-NN`, Tabel E booklet). Pencocokan nama kota/kecamatan bersifat longgar karena penulisan berbeda antar-tabel.

In [4]:
m = P.muat_master()
rel0, pas = P.cek_relasi()          # pemeriksaan kunci (R26–R27 menyusul setelah profil stasiun pada bagian 4)
display(rel0[["kode", "status", "jumlah_tidak_cocok", "pemeriksaan", "detail"]].style.map(
    lambda v: {"OK": "background-color:#D6F0E0", "MASALAH": "background-color:#F8D0CC", "INFO": "background-color:#E3EEF9"}.get(v, ""), subset=["status"]))

,kode,status,jumlah_tidak_cocok,pemeriksaan,detail
0,R01,OK,0,sites.location_id unik,
1,R02,OK,0,sensors.sensor_id unik,
2,R03,OK,0,hospitals.hospital_id unik,
3,R04,OK,0,sensors.location_id ada di sites.location_id,
4,R05,OK,0,setiap lokasi di sites.csv punya sensor,
5,R06,OK,0,metadata_dates.location_id sama dengan sites.location_id,
6,R07,OK,0,tiap file AQMS memuat tepat 1 sensor_id,{}
7,R08,OK,0,sensor_id di file AQMS terdaftar di sensors.csv,{}
8,R09,OK,0,kode di nama file (LOC-) cocok dengan sensor_id di dalam file,
9,R10,OK,0,tidak ada sensor_id yang dipakai dua file,


In [5]:
print("Pasangan 27 stasiun ↔ sensor ↔ lokasi ↔ RS:")
display(pas[["nomor", "kode_stasiun", "sensor_id", "location_id", "hospital_id", "nama_rumah_sakit", "provinsi", "pulau",
             "kelompok_lokasi", "kapasitas_tempat_tidur", "status_operasional", "kota_cocok", "kecamatan_cocok"]])
print("\nJumlah stasiun per pulau × kelompok lokasi (dasar analisis jaringan, Tujuan 5):")
display(pd.crosstab(pas.pulau, pas.kelompok_lokasi).reindex(index=cfg.URUTAN_PULAU, columns=cfg.URUTAN_KELOMPOK, fill_value=0).assign(Total=lambda d: d.sum(axis=1)))

Pasangan 27 stasiun ↔ sensor ↔ lokasi ↔ RS:


,nomor,kode_stasiun,sensor_id,location_id,hospital_id,nama_rumah_sakit,provinsi,pulau,kelompok_lokasi,kapasitas_tempat_tidur,status_operasional,kota_cocok,kecamatan_cocok
0,1,JKT-01,SENS-001,LOC-JKT-01,RS-XXX-01,RSUD XXX Gambir,DKI Jakarta,Jawa,Perkotaan,300.0,Active,True,True
1,2,JKT-02,SENS-002,LOC-JKT-02,RS-XXX-02,RSUD XXX Cilincing,DKI Jakarta,Jawa,Industri,250.0,Active,True,True
2,3,JKT-03,SENS-003,LOC-JKT-03,RS-XXX-03,RSUD XXX Cengkareng,DKI Jakarta,Jawa,Jalan Raya,280.0,Active,True,True
3,4,KTG-01,SENS-004,LOC-KTG-01,RS-XXX-04,RSUD XXX Jekan Raya,Kalimantan Tengah,Kalimantan,Perkotaan,220.0,Active,True,True
4,5,KTG-02,SENS-005,LOC-KTG-02,RS-XXX-05,RSUD XXX Mentawa Baru,Kalimantan Tengah,Kalimantan,Gambut,150.0,Active,True,True
5,6,BGR-01,SENS-006,LOC-BGR-01,RS-XXX-06,RS Paru XXX Cisarua,Jawa Barat,Jawa,Latar Bersih,200.0,Active,True,True
6,7,BGR-02,SENS-007,LOC-BGR-02,RS-XXX-07,RSUD XXX Bogor Tengah,Jawa Barat,Jawa,Jalan Raya,260.0,Active,True,True
7,8,YKY-01,SENS-008,LOC-YKY-01,RS-XXX-08,RSUD XXX Sleman,D.I. Yogyakarta,Jawa,Perkotaan,250.0,Active,True,True
8,9,IDP-01,SENS-009,LOC-IDP-01,RS-XXX-09,RSUD XXX Indrapuri,Aceh,Sumatera,Latar Bersih,120.0,Active,True,True
9,10,KTB-01,SENS-010,LOC-KTB-01,RS-XXX-10,RSUD XXX Agam,Sumatera Barat,Sumatera,Latar Bersih,180.0,Active,True,True



Jumlah stasiun per pulau × kelompok lokasi (dasar analisis jaringan, Tujuan 5):


kelompok_lokasi,Perkotaan,Jalan Raya,Industri,Gambut,Latar Bersih,Total
pulau,,,,,,
Sumatera,3,0,1,4,2,10
Jawa,3,2,1,0,1,7
Kalimantan,1,0,1,4,0,6
Sulawesi,0,0,1,0,1,2
Papua,0,0,0,0,2,2


## 4. Profil kualitas per stasiun (27 file AQMS)
Definisi pemeriksaan:

| Pemeriksaan | Definisi |
|---|---|
| Missing | String kosong atau token `NaN/null/none/N/A/-` |
| Non-numerik | Teks lain yang tidak bisa dibaca sebagai angka berhingga (mis. `0xDEADBEEF`, `#VALUE!`, `INF`) |
| Nilai mustahil | Sentinel (`±999`, `±9999`, `-9999`, `99999`), polutan negatif, suhu <5 atau >45 °C, RH di luar 0–100 %, angin >40 m/s, arah di luar 0–360° |
| Nilai ekstrem | PM2.5 atau PM10 >500 µg/m³ (batas atas tabel ISPU). **Belum tentu salah**; bisa episode asap nyata |
| Timestamp bergeser | `timestamp_start` tidak tepat di awal jam; geseran bertanda (− = sebelum jam penuh, mis. 23:48 → −12 menit) |
| Jam hilang | Jam di antara 24.432 jam yang diharapkan yang tidak punya satu pun baris (setelah pembulatan ke jam terdekat) |
| Duplikat per jam | Baris tambahan pada jam yang sama setelah pembulatan |

Catatan: `999` pada **CO** tidak dianggap sentinel karena 999 µg/m³ adalah konsentrasi CO yang wajar (lihat `config.SENTINEL_KECUALI`).

In [6]:
prof = P.profil_semua_stasiun(verbose=False)
ring = prof["ringkasan"]
rel, pas = P.cek_relasi(ringkasan=ring)       # kini lengkap dengan R26–R27 (metadata_dates & jumlah baris)
urutan = P.urutan_stasiun(pas)
print(f"{len(ring)} stasiun diprofilkan, total {rp(ring.n_baris.sum())} baris")
display(rel[rel.kode.isin(["R26", "R27"])][["kode", "status", "jumlah_tidak_cocok", "pemeriksaan", "detail"]])

27 stasiun diprofilkan, total 667.981 baris


,kode,status,jumlah_tidak_cocok,pemeriksaan,detail
25,R26,OK,0,metadata_dates (awal–akhir) = 2024-01-01 00:00 s.d. 2026-10-14 23:00,
26,R27,MASALAH,2,jumlah baris file = 24.432 (sesuai resolusi 1 jam di metadata_dates),MDN-01=20800; PLB-01=36381


### 4.1 Timestamp, jam hilang, dan duplikat

In [7]:
kol_waktu = ["n_baris", "selisih_baris", "n_ts_luar_jendela", "n_ts_bergeser", "persen_ts_bergeser", "geser_median_menit", "geser_min_menit",
             "geser_maks_menit", "n_geser_tepat_30menit", "n_durasi_bukan_1jam", "n_jam_hilang", "persen_jam_hilang", "gap_terpanjang_jam",
             "n_baris_lebih_per_jam", "n_jam_duplikat", "n_jam_duplikat_konflik", "n_ts_duplikat_persis", "n_id_duplikat", "n_baris_identik"]
t = ring.set_index("kode").loc[urutan, kol_waktu]
display(tandai_positif(t, [c for c in kol_waktu if c not in ("n_baris", "geser_median_menit", "geser_min_menit", "geser_maks_menit")]))

print("\nSebaran offset menit (dari awal jam) pada stasiun yang timestamp-nya bergeser:")
display(ring[ring.n_ts_bergeser > 0].set_index("kode")[["n_ts_bergeser", "sebaran_offset"]])
print("\nStasiun dengan jendela < 1 jam (resolusi lebih halus dari nominal):")
display(ring[ring.n_durasi_subjam > 0][["kode", "n_durasi_subjam", "durasi_subjam_menit", "subjam_awal", "subjam_akhir"]])
print("\nBlok jam berurutan yang memuat baris sub-jam pada PLB-01 (rentang awal–akhir di atas hanya batas luar, bukan periode kontigu):")
blok_plb = P.periode_subjam("PLB-01")
display(blok_plb)
ms_st = P.format_milidetik()
print(f"\nFile dengan format timestamp berpecahan detik (.000): {len(ms_st)} dari {len(ring)} stasiun")

,n_baris,selisih_baris,n_ts_luar_jendela,n_ts_bergeser,persen_ts_bergeser,geser_median_menit,geser_min_menit,geser_maks_menit,n_geser_tepat_30menit,n_durasi_bukan_1jam,n_jam_hilang,persen_jam_hilang,gap_terpanjang_jam,n_baris_lebih_per_jam,n_jam_duplikat,n_jam_duplikat_konflik,n_ts_duplikat_persis,n_id_duplikat,n_baris_identik
kode,,,,,,,,,,,,,,,,,,,
BGL-01,24432,0,0,24432,100.000000,2.170000,2.170000,2.170000,0,0,0,0.000000,0,0,0,0,0,0,0
MDN-01,20800,-3632,0,20800,100.000000,4.300000,4.300000,4.300000,0,0,3632,14.870000,121,0,0,0,0,0,0
PSW-01,24432,0,0,24432,100.000000,3.760000,3.760000,3.760000,0,0,0,0.000000,0,0,0,0,0,0,0
BTM-01,24432,0,0,24432,100.000000,-11.750000,-11.750000,-11.750000,0,0,0,0.000000,0,0,0,0,0,0,0
JMB-01,24432,0,0,24432,100.000000,-4.490000,-4.490000,-4.490000,0,0,0,0.000000,0,0,0,0,0,0,0
JMB-02,24432,0,132,24300,99.460000,5.200000,5.200000,5.200000,0,92,224,0.920000,2,92,92,86,223,0,11
PKU-01,24432,0,0,24432,100.000000,0.000000,0.000000,0.000000,0,0,0,0.000000,0,0,0,0,0,0,0
PLB-01,36381,11949,0,11949,32.840000,15.000000,-15.000000,30.000000,3983,15933,0,0.000000,0,11949,3986,3956,0,0,0
IDP-01,24432,0,0,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0.000000,0,0,0,0,0,0,0



Sebaran offset menit (dari awal jam) pada stasiun yang timestamp-nya bergeser:


,n_ts_bergeser,sebaran_offset
kode,,
BGL-01,24432,+2 mnt: 24432
BJB-01,24432,+1 mnt: 24432
BTM-01,24432,+48 mnt: 24432
JMB-01,24432,+56 mnt: 24432
JMB-02,24300,+5 mnt: 24300
KTB-01,24432,+1 mnt: 24432
MDN-01,20800,+4 mnt: 20800
MER-01,24432,+5 mnt: 24432
MLG-01,24432,+58 mnt: 24432



Stasiun dengan jendela < 1 jam (resolusi lebih halus dari nominal):


,kode,n_durasi_subjam,durasi_subjam_menit,subjam_awal,subjam_akhir
10,JMB-02,92,-60,2024-01-30 19:05:12.100,2026-10-06 19:05:12.100
20,PLB-01,15933,15,2024-09-01 00:00:00.000,2026-10-14 23:00:00.000



Blok jam berurutan yang memuat baris sub-jam pada PLB-01 (rentang awal–akhir di atas hanya batas luar, bukan periode kontigu):


,blok,awal,akhir,jumlah_jam,baris_subjam
0,1,2024-09-01,2024-10-31 23:00:00,1464,5856
1,2,2025-09-01,2025-10-31 23:00:00,1464,5856
2,3,2026-09-01,2026-10-14 23:00:00,1056,4221



File dengan format timestamp berpecahan detik (.000): 18 dari 27 stasiun


### 4.2 Nilai kosong, teks galat, nilai mustahil, dan nilai ekstrem

In [8]:
kol_nilai = ["n_baris_kosong", "n_baris_teks", "n_baris_mustahil", "n_baris_ekstrem", "n_pm25_gt_pm10", "n_valid_tapi_mustahil", "n_valid_tapi_kosong"]
display(tandai_positif(ring.set_index("kode").loc[urutan, kol_nilai], kol_nilai))

print("\nRincian per kolom (hanya stasiun/kolom yang bermasalah):")
kb = prof["kolom"]
kb = kb[(kb.n_kosong > 0) | (kb.n_teks > 0) | (kb.n_mustahil > 0) | (kb.n_ekstrem > 0)]
display(kb.drop(columns=["n", "n_numerik"]).reset_index(drop=True))

,n_baris_kosong,n_baris_teks,n_baris_mustahil,n_baris_ekstrem,n_pm25_gt_pm10,n_valid_tapi_mustahil,n_valid_tapi_kosong
kode,,,,,,,
BGL-01,0,0,0,0,0,0,0
MDN-01,624,0,0,0,0,0,0
PSW-01,0,0,0,0,0,0,0
BTM-01,0,0,0,0,0,0,0
JMB-01,0,0,0,0,0,0,0
JMB-02,7519,8997,24308,4897,4897,0,0
PKU-01,0,0,0,0,0,0,0
PLB-01,5136,0,0,0,0,0,0
IDP-01,0,0,0,0,0,0,0



Rincian per kolom (hanya stasiun/kolom yang bermasalah):


,kode,kolom,n_kosong,n_teks,n_sentinel,n_negatif,n_luar_rentang,n_mustahil,n_ekstrem
0,JMB-02,pm25_ug_m3,4661,7163,891,6698,0,7589,4897
1,JMB-02,pm10_ug_m3,4671,7156,889,6693,0,7582,0
2,JMB-02,so2_ug_m3,4626,7114,976,6693,0,7669,0
3,JMB-02,co_ug_m3,4632,7206,7571,0,0,7571,0
4,JMB-02,o3_ug_m3,4633,7149,5835,6693,0,12528,0
5,JMB-02,no2_ug_m3,4639,7110,967,6693,0,7660,0
6,JMB-02,temperature_c,3718,124,8606,0,11862,20468,0
7,JMB-02,humidity_pct,3718,149,0,0,18255,18255,0
8,JMB-02,wind_speed_ms,3718,0,0,0,20498,20498,0
9,JMB-02,wind_direction_deg,0,0,15597,0,8619,24216,0


In [9]:
print("Jenis teks galat pada kolom numerik (non-numerik):")
tok = prof["token"].copy()
tok["jenis"] = tok.token.map(P.jenis_token)
ringkas_tok = tok.groupby(["kode", "jenis", "token"]).n.sum().reset_index().sort_values(["kode", "n"], ascending=[True, False])
display(ringkas_tok)
print("\nJumlah sel teks galat per kolom, JMB-02:")
display(tok[tok.kode == "JMB-02"].groupby("kolom").n.sum().to_frame("sel_teks_galat").T)

Jenis teks galat pada kolom numerik (non-numerik):


,kode,jenis,token,n
10,JMB-02,kode heksadesimal (isi memori),0xDEADBEEF,5706
3,JMB-02,galat komunikasi/integritas data,ERR_COMM_TIMEOUT,5562
2,JMB-02,galat komunikasi/integritas data,BAD_CHECKSUM,5108
8,JMB-02,galat perangkat keras sensor,VOLT_SURGE_99V,5072
1,JMB-02,galat formula spreadsheet,#VALUE!,5042
5,JMB-02,galat perangkat keras sensor,ADC_SAT_FAIL,5041
11,JMB-02,nilai tak hingga,INF,5032
7,JMB-02,galat perangkat keras sensor,TAPE_TEAR_CRITICAL,4983
0,JMB-02,byte kosong/null,NULL_BYTE,460
4,JMB-02,galat perangkat keras sensor,ADC_SATURATED,460



Jumlah sel teks galat per kolom, JMB-02:


kolom,co_ug_m3,humidity_pct,no2_ug_m3,o3_ug_m3,pm10_ug_m3,pm25_ug_m3,so2_ug_m3,temperature_c
sel_teks_galat,7206,149,7110,7149,7156,7163,7114,124


In [10]:
print("Nilai sentinel (999 / -999 / 9999 / -9999) menurut qc_flag:")
sq = prof["sentinel_qc"]
display(sq.pivot_table(index=["kode", "kolom", "nilai"], columns="qc_flag", values="n", aggfunc="sum", fill_value=0))
print("\nBaris ber-nilai mustahil menurut qc_flag (apakah penanda QC alat menangkapnya?):")
display(prof["mustahil_qc"].pivot_table(index="kode", columns="qc_flag", values="n", aggfunc="sum", fill_value=0))

Nilai sentinel (999 / -999 / 9999 / -9999) menurut qc_flag:


qc_flag                            COMM_LOST  CORRUPT  FATAL_ERR  GARBAGE  PARSER_FAIL  SENSOR_DEAD  SUSPECT
kode   kolom              nilai                                                                             
JMB-02 co_ug_m3           -9999.0        113      134        139      142          124          112      114
                          -999.0         978      929        921      942         1011          939      973
       no2_ug_m3          -9999.0        135      134        141      141          151          139      126
       o3_ug_m3           -9999.0        120      127        154      142          127          134      130
                           9999.0        702      707        679      718          734          710      651
       pm10_ug_m3         -9999.0        112      126        143      158          125          124      101
       pm25_ug_m3         -9999.0        112      130        141      155          128          124      101
       so2_ug_m3          -9999.0        122      132        151      165          154          131      121
       temperature_c      -999.0        1202     1184       1277     1228         1172         1225     1186
                           999.0           0       33         32       30           37            0        0
       wind_direction_deg -999.0        1202     1184       1277     1228         1172         1225     1186
                           9999.0        978     1010       1035     1042         1146          939      973
SM2-01 pm10_ug_m3          999.0           0        0          0        0            0            0     1039
       pm25_ug_m3          999.0           0        0          0        0            0            0     1039


Baris ber-nilai mustahil menurut qc_flag (apakah penanda QC alat menangkapnya?):


qc_flag,COMM_LOST,CORRUPT,FATAL_ERR,GARBAGE,PARSER_FAIL,SENSOR_DEAD,SUSPECT
kode,,,,,,,
JMB-02,3433,3425,3545,3536,3633,3389,3347
MER-01,0,0,0,0,0,0,2529
SM2-01,0,0,0,0,0,0,1039


### 4.3 Distribusi `qc_flag`
Label di luar spesifikasi `metadata_sensors.csv` (VALID, SUSPECT, CALIBRATING, MISSING) ditandai sebagai *non-standar*.

In [11]:
qc = prof["qc"]
pv = qc.pivot_table(index="kode", columns="qc_flag", values="persen", aggfunc="sum", fill_value=0).reindex(urutan)
urut_flag = ["VALID", "SUSPECT", "CALIBRATING", "WARMUP", "MAINTENANCE", "MISSING"] + [c for c in pv.columns if c in cfg.QC_GALAT]
urut_flag += [c for c in pv.columns if c not in urut_flag]
print("Persentase baris per label qc_flag (%):")
display(pv[urut_flag].round(2).style.background_gradient(cmap="Blues", axis=None))
print("\nSemua label yang ditemukan:", sorted(qc.qc_flag.unique()))
print("Label di luar spesifikasi metadata:", sorted(set(qc.qc_flag) - set(cfg.QC_STANDAR)))

fig = P.fig_qc_distribusi(qc, urutan, "Distribusi qc_flag per stasiun (% baris)")
style.simpan_html(fig, "02_qc_flag_distribusi")
fig.show()

Persentase baris per label qc_flag (%):


qc_flag,VALID,SUSPECT,CALIBRATING,WARMUP,MAINTENANCE,MISSING,COMM_LOST,CORRUPT,FATAL_ERR,GARBAGE,PARSER_FAIL,SENSOR_DEAD
kode,,,,,,,,,,,,
BGL-01,99.490000,0.510000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
MDN-01,80.000000,8.000000,5.000000,4.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PSW-01,99.440000,0.560000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
BTM-01,99.370000,0.630000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
JMB-01,99.310000,0.690000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
JMB-02,0.500000,13.700000,0.000000,0.000000,0.000000,0.000000,14.050000,14.020000,14.510000,14.480000,14.870000,13.870000
PKU-01,99.460000,0.540000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PLB-01,75.440000,6.050000,4.390000,0.000000,0.000000,14.120000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
IDP-01,99.550000,0.450000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000



Semua label yang ditemukan: ['CALIBRATING', 'COMM_LOST', 'CORRUPT', 'FATAL_ERR', 'GARBAGE', 'MAINTENANCE', 'MISSING', 'PARSER_FAIL', 'SENSOR_DEAD', 'SUSPECT', 'VALID', 'WARMUP']
Label di luar spesifikasi metadata: ['COMM_LOST', 'CORRUPT', 'FATAL_ERR', 'GARBAGE', 'MAINTENANCE', 'PARSER_FAIL', 'SENSOR_DEAD', 'WARMUP']


### 4.4 Heatmap kelengkapan data (stasiun × bulan)
Urutan stasiun: pulau → kelompok lokasi. **Gelap = lengkap.** *Jam VALID* menghitung jam yang punya baris `VALID` dengan PM2.5 numerik wajar;
*jam tercatat* hanya menghitung adanya baris.

In [12]:
bul = prof["bulanan"]
fig = P.fig_heatmap_ketersediaan(bul, urutan, "jam_valid", "Kelengkapan data VALID per stasiun × bulan (% dari jam dalam bulan)")
style.simpan_html(fig, "02_kelengkapan_valid_heatmap")
fig.show()

In [13]:
fig = P.fig_heatmap_ketersediaan(bul, urutan, "jam_tercatat", "Jam tercatat (ada baris, apa pun statusnya) per stasiun × bulan (%)")
style.simpan_html(fig, "02_kelengkapan_tercatat_heatmap")
fig.show()

# Kelengkapan VALID per stasiun (ringkas), dari terendah
kelengkapan = ring.set_index("kode")[["n_jam_unik", "persen_jam_hilang", "n_jam_valid", "persen_jam_valid"]].sort_values("persen_jam_valid")
display(kelengkapan.head(8))

,n_jam_unik,persen_jam_hilang,n_jam_valid,persen_jam_valid
kode,,,,
JMB-02,24208,0.92,121,0.50
MER-01,24432,0.00,4891,20.02
MDN-01,20800,14.87,16640,68.11
PLB-01,24432,0.00,19412,79.45
SM2-01,24432,0.00,20767,85.00
JKT-01,24432,0.00,23454,96.00
JKT-03,24432,0.00,23470,96.06
JKT-02,24432,0.00,23480,96.10


## 5. Statistik deskriptif awal (sebelum dibersihkan) & penandaan stasiun mencurigakan
Statistik dihitung pada **semua nilai numerik apa adanya** (termasuk sentinel dan non-VALID), dalam **satuan label file**.
Kolom `*_est_ugm3` memakai faktor hipotesis ppm → µg/m³ **hanya untuk membandingkan skala** dengan `baseline_pm25`.

In [14]:
d = prof["desk"]
satuan_st = ring.set_index("kode").satuan

def tabel_desk(kanon, kolom_stat=("n", "min", "p05", "median", "rata2", "p95", "p99", "maks")):
    x = d[d.kanon == kanon].set_index("kode")[list(kolom_stat)].reindex(urutan)
    x.insert(0, "satuan_label", satuan_st.reindex(urutan).values)
    return x.round(4)

print("PM2.5 — statistik mentah (satuan label file):")
display(tabel_desk("pm25"))

PM2.5 — statistik mentah (satuan label file):


,satuan_label,n,min,p05,median,rata2,p95,p99,maks
kode,,,,,,,,,
BGL-01,ug/m3,24432,9.7000,15.4000,25.1000,25.5714,40.5000,49.6000,65.0000
MDN-01,ppm,20176,0.0000,0.0149,0.0469,0.0506,0.1126,0.1840,0.2692
PSW-01,ppm,24432,0.0177,0.0269,0.0435,0.0507,0.0957,0.1120,0.1335
BTM-01,ug/m3,24432,16.6000,24.2000,38.5000,47.5765,98.5000,121.5690,191.5000
JMB-01,ug/m3,24432,12.0000,20.1000,32.3000,44.4595,109.4000,126.5000,204.1000
JMB-02,ug/m3,12608,-9999.0000,-9999.0000,-278.8850,20418.9257,89135.6390,97746.3031,99954.0700
PKU-01,ug/m3,24432,15.5000,22.1000,35.3000,47.0935,110.6000,130.0140,201.3000
PLB-01,ug/m3,31245,15.9000,27.7000,72.8000,89.6967,187.9000,236.9800,285.1000
IDP-01,ug/m3,24432,2.0000,4.0000,8.9000,9.3374,16.3000,20.0000,24.5000


In [15]:
print("PM10 — statistik mentah:")
display(tabel_desk("pm10"))
print("\nMedian polutan gas (satuan label file; stasiun ppm tidak sebanding dengan µg/m³ sebelum konversi):")
med = d.pivot_table(index="kode", columns="kanon", values="median").reindex(urutan)[["so2", "co", "o3", "no2"]]
med.insert(0, "satuan_label", satuan_st.reindex(urutan).values)
display(med.round(4))
print("\nCuaca — min / median / maks per stasiun (cek kewajaran fisik):")
cu = d[d.kanon.isin(cfg.CUACA)].pivot_table(index="kode", columns="kanon", values=["min", "median", "maks"]).reindex(urutan)
cu.columns = [f"{b}_{a}" for a, b in cu.columns]
display(cu[[c for c in cu.columns if c.startswith(("temperature_c", "humidity_pct", "wind_speed_ms"))]].round(1))

PM10 — statistik mentah:


,satuan_label,n,min,p05,median,rata2,p95,p99,maks
kode,,,,,,,,,
BGL-01,ug/m3,24432,13.9000,22.8000,37.4000,38.2428,60.9000,74.6690,101.5000
MDN-01,ppm,20176,0.0000,0.0222,0.0699,0.0740,0.1576,0.2576,0.3769
PSW-01,ppm,24432,0.0240,0.0381,0.0621,0.0726,0.1378,0.1610,0.1984
BTM-01,ug/m3,24432,23.4000,35.7000,57.4000,70.1314,143.2000,156.8000,239.4000
JMB-01,ug/m3,24432,17.1000,29.8000,48.2000,63.7385,139.0000,158.5000,255.1000
JMB-02,ug/m3,12605,-9999.0000,-9999.0000,-560.5500,-1393.9493,0.0010,0.0010,124.2400
PKU-01,ug/m3,24432,22.6000,32.9000,52.8000,67.5487,140.3000,162.4830,251.6000
PLB-01,ug/m3,31245,23.1000,39.4000,103.0000,115.1139,229.1000,289.0360,347.7000
IDP-01,ug/m3,24432,4.5000,6.7000,14.9000,15.6242,27.2000,33.6000,43.3000



Median polutan gas (satuan label file; stasiun ppm tidak sebanding dengan µg/m³ sebelum konversi):


kanon,satuan_label,so2,co,o3,no2
kode,,,,,
BGL-01,ug/m3,7.1000,350.000,10.4000,18.1000
MDN-01,ppm,0.0034,0.564,0.0130,0.0185
PSW-01,ppm,0.0029,0.369,0.0043,0.0104
BTM-01,ug/m3,7.2000,349.800,10.4000,18.1000
JMB-01,ug/m3,7.2000,350.000,10.4000,18.1000
JMB-02,ug/m3,-99.0000,-999.000,-50.0000,-40.0000
PKU-01,ug/m3,7.1000,349.800,10.4500,18.1000
PLB-01,ug/m3,8.5000,467.000,8.5000,21.2000
IDP-01,ug/m3,4.5000,161.400,12.6000,10.8000



Cuaca — min / median / maks per stasiun (cek kewajaran fisik):


,humidity_pct_maks,temperature_c_maks,wind_speed_ms_maks,humidity_pct_median,temperature_c_median,wind_speed_ms_median,humidity_pct_min,temperature_c_min,wind_speed_ms_min
kode,,,,,,,,,
BGL-01,93.1,30.5,4.5,85.0,24.5,2.5,60.0,22.4,0.4
MDN-01,99.0,33.3,5.4,92.0,24.5,2.2,56.8,22.0,0.4
PSW-01,93.6,31.0,3.4,86.0,25.0,2.1,61.0,23.0,0.9
BTM-01,92.8,30.5,4.4,85.0,24.5,2.5,60.0,22.6,0.4
JMB-01,93.2,30.5,4.3,85.0,24.5,2.5,60.0,22.3,0.4
JMB-02,999.9,1450.5,350.0,999.9,-273.2,-99.0,-150.0,-999.0,-99.9
PKU-01,92.1,30.5,4.4,85.0,24.5,2.5,60.0,22.6,0.4
PLB-01,93.9,31.0,3.6,85.0,25.0,1.9,60.0,22.9,0.3
IDP-01,94.3,30.5,5.9,85.9,25.0,3.1,62.1,23.0,0.8


### 5.1 Pemeriksaan silang: pola musiman & PM2.5 vs kasus RS
Dua pemeriksaan deskriptif (bukan inferensi) dipakai untuk menandai stasiun yang perlu dicurigai, memakai PM2.5 harian pendekatan dari baris `VALID` non-mustahil
(≥12 jam per hari; stasiun ppm dikalikan 1000). Profil kesehatan dihitung lebih dulu di sini; hasil lengkapnya dibahas di Bagian 7.

In [16]:
kes = P.profil_kesehatan()                       # dipakai juga pada Bagian 7
kel_lokasi = pas.set_index("kode_stasiun").kelompok_lokasi
musiman = P.korelasi_musiman(prof["harian"], kel_lokasi, grup="Gambut", kecualikan=("JMB-02",))
silang = P.silang_pm25_kasus(prof["harian"], kes["raw"], pas)

print("Korelasi profil musiman bulanan PM2.5 stasiun gambut terhadap rata-rata stasiun gambut lain (JMB-02 dikecualikan: data tidak layak):")
display(musiman)
print("\nPM2.5 rata-rata vs kasus RS pasangan (peringkat 1 = tertinggi; selisih positif = kasus RS lebih rendah dari yang 'diharapkan' menurut PM2.5):")
display(silang.round(2).sort_values("selisih_peringkat", ascending=False))

Korelasi profil musiman bulanan PM2.5 stasiun gambut terhadap rata-rata stasiun gambut lain (JMB-02 dikecualikan: data tidak layak):


,kode,korelasi_musiman_vs_rekan,bulan_puncak,pm25_bulan_puncak,pm25_bulan_terendah
0,BJB-01,0.26,12,186.5,25.3
1,KTG-02,0.86,10,234.8,21.9
2,PKU-01,0.96,9,92.0,25.5
3,JMB-01,0.96,9,89.7,23.3
4,STG-01,0.97,9,91.0,16.0
5,PLB-01,0.97,9,140.7,31.7
6,PKN-01,0.98,10,119.9,17.4



PM2.5 rata-rata vs kasus RS pasangan (peringkat 1 = tertinggi; selisih positif = kasus RS lebih rendah dari yang 'diharapkan' menurut PM2.5):


,kode,hospital_id,n_hari_bersama,pm25_rata2,resp_rata2_per_hari,u5_rata2_per_hari,kapasitas_tt,resp_per_100_tt,spearman_pm25_resp,peringkat_pm25,peringkat_resp,selisih_peringkat
25,BJB-01,RS-XXX-26,1018,102.14,10.49,3.36,300.0,3.50,0.19,1.0,25.0,24.0
24,SM2-01,RS-XXX-25,1008,88.20,13.93,4.47,600.0,2.32,0.45,2.0,24.0,22.0
23,MDN-01,RS-XXX-24,981,45.58,14.58,4.66,720.0,2.02,-0.19,12.0,22.0,10.0
4,KTG-02,RS-XXX-05,1018,61.06,42.30,9.57,150.0,28.20,0.53,4.0,6.0,2.0
20,PLB-01,RS-XXX-21,1015,65.64,42.53,12.51,220.0,19.33,0.87,3.0,5.0,2.0
10,PKU-01,RS-XXX-11,1018,47.09,28.04,7.97,180.0,15.58,0.86,11.0,11.0,0.0
22,MER-01,RS-XXX-23,54,16.13,14.10,2.95,150.0,9.40,-0.06,23.0,23.0,0.0
12,BTM-01,RS-XXX-13,1018,47.57,28.42,8.06,180.0,15.79,0.86,10.0,10.0,0.0
13,BGL-01,RS-XXX-14,1018,25.57,18.98,4.49,180.0,10.55,0.42,19.0,19.0,0.0
8,IDP-01,RS-XXX-09,1018,9.34,8.25,1.80,120.0,6.88,0.10,26.0,26.0,0.0


In [17]:
cur = P.tanda_curiga(ring, prof["desk"], m["sites"], musiman=musiman, silang=silang)
print(f"{int(cur.mencurigakan.sum())} dari {len(cur)} stasiun ditandai mencurigakan. Aturan: lihat docstring P.tanda_curiga.")
display(cur[cur.mencurigakan].sort_values("jumlah_tanda", ascending=False)[
    ["kode", "jumlah_tanda", "median_pm25_est", "maks_pm25_est", "baseline_pm25", "rasio_median_baseline", "alasan"]].round(2))
print("\nStasiun tanpa tanda kecurigaan:", ", ".join(cur[~cur.mencurigakan].kode))

10 dari 27 stasiun ditandai mencurigakan. Aturan: lihat docstring P.tanda_curiga.


,kode,jumlah_tanda,median_pm25_est,maks_pm25_est,baseline_pm25,rasio_median_baseline,alasan
10,JMB-02,6,-278.88,99954.07,68.9,-4.05,VALID hanya 0.5%; ada teks galat; ada nilai mustahil; median PM2.5 -4.0× baseline; PM2.5 maks 99.954 µg/m³; PM2.5 > ...
14,MDN-01,4,46.90,269.20,42.8,1.10,satuan ppm; jumlah baris 20800 ≠ 24432; VALID hanya 80.0%; PM2.5 tinggi tetapi kasus RS rendah (peringkat kasus 10 t...
15,MER-01,4,15.26,1449.33,16.5,0.92,VALID hanya 20.0%; ada nilai mustahil; PM2.5 maks 1.449 µg/m³; PM2.5 > PM10 >1% baris
23,SM2-01,4,48.62,999.00,32.5,1.50,VALID hanya 85.0%; ada nilai mustahil; PM2.5 maks 999 µg/m³; PM2.5 tinggi tetapi kasus RS rendah (peringkat kasus 22...
3,BJB-01,3,45.10,336.70,35.0,1.29,satuan ppm; pola musiman menyimpang (r=0.26 vs stasiun gambut lain); PM2.5 tinggi tetapi kasus RS rendah (peringkat ...
20,PLB-01,2,72.80,285.10,42.0,1.73,jumlah baris 36381 ≠ 24432; VALID hanya 75.4%
12,KTG-01,1,24.80,1620.50,22.0,1.13,PM2.5 maks 1.620 µg/m³
13,KTG-02,1,24.89,1530.69,35.0,0.71,PM2.5 maks 1.531 µg/m³
18,PKN-01,1,24.30,646.10,24.0,1.01,PM2.5 maks 646 µg/m³
22,PSW-01,1,43.50,133.50,54.3,0.80,satuan ppm



Stasiun tanpa tanda kecurigaan: BGL-01, BGR-01, BGR-02, BTM-01, IDP-01, JKT-01, JKT-02, JKT-03, JMB-01, KTB-01, MLG-01, MRS-01, PKU-01, PLU-01, SRG-01, STG-01, YKY-01


## 6. Verifikasi hipotesis satuan ppm dan masalah yang sudah diketahui
Hipotesis konversi diuji dengan **orde besaran**: median terkonversi dibandingkan dengan (a) `baseline_pm25` stasiun yang sama untuk PM2.5, dan
(b) median stasiun berunit µg/m³ untuk polutan lain. `faktor_implisit` = faktor yang membuat median tepat sama dengan acuan (pembanding terhadap faktor hipotesis).
Ini uji kewajaran, bukan bukti pasti.

In [18]:
ppm = P.verifikasi_ppm(prof["desk"], ring, m["sites"])
display(ppm)
print(f"\nHipotesis didukung pada {int(ppm.hipotesis_didukung.sum())} dari {len(ppm)} kombinasi stasiun × polutan.")
print("Rasio median terkonversi/acuan, PM2.5 saja:", ppm[ppm.polutan == 'pm25'].set_index('kode').rasio.to_dict())

,kode,polutan,median_label_ppm,faktor_hipotesis,median_terkonversi_ugm3,acuan_ugm3,jenis_acuan,rasio,faktor_implisit,hipotesis_didukung
0,BJB-01,pm25,0.0451,1000.00,45.10,35.00,baseline_pm25 stasiun,1.29,776.1,True
1,BJB-01,pm10,0.0653,1000.00,65.30,41.15,median antar-stasiun µg/m³,1.59,630.2,True
2,BJB-01,so2,0.0032,2620.45,8.39,7.20,median antar-stasiun µg/m³,1.16,2250.0,True
3,BJB-01,co,0.4750,1145.60,544.16,350.40,median antar-stasiun µg/m³,1.55,737.7,True
4,BJB-01,o3,0.0090,1963.19,17.67,10.50,median antar-stasiun µg/m³,1.68,1166.7,True
5,BJB-01,no2,0.0164,1881.80,30.86,18.10,median antar-stasiun µg/m³,1.71,1103.7,True
6,MDN-01,pm25,0.0469,1000.00,46.90,42.80,baseline_pm25 stasiun,1.10,912.6,True
7,MDN-01,pm10,0.0699,1000.00,69.90,41.15,median antar-stasiun µg/m³,1.70,588.7,True
8,MDN-01,so2,0.0034,2620.45,8.91,7.20,median antar-stasiun µg/m³,1.24,2117.6,True
9,MDN-01,co,0.5640,1145.60,646.12,350.40,median antar-stasiun µg/m³,1.84,621.3,True



Hipotesis didukung pada 18 dari 18 kombinasi stasiun × polutan.
Rasio median terkonversi/acuan, PM2.5 saja: {'BJB-01': 1.29, 'MDN-01': 1.1, 'PSW-01': 0.8}


In [19]:
rs = ring.set_index("kode")
dd = prof["desk"]; kb_all = prof["kolom"]; tkn = prof["token"]; sq = prof["sentinel_qc"]
klaim = []
def tambah_klaim(k, bukti, status):
    klaim.append(dict(klaim=k, bukti_terukur=bukti, status=status))

# 1–2. satuan ppm
ppm_st = sorted(rs[rs.satuan == "ppm"].index)
tambah_klaim("BJB-01, MDN-01, PSW-01 memakai kolom *_ppm", f"Stasiun berkolom *_ppm: {', '.join(ppm_st)}; sites.pm25_unit konsisten (cek R16/R17)",
             "Terkonfirmasi" if ppm_st == ["BJB-01", "MDN-01", "PSW-01"] else "Berbeda dari klaim")
n_ok, n_tot = int(ppm.hipotesis_didukung.sum()), len(ppm)
tambah_klaim("Konversi PM ×1000 dan gas ppm×BM/24,45×1000", f"{n_ok}/{n_tot} kombinasi stasiun×polutan masuk rentang kewajaran orde besaran terhadap acuan",
             "Didukung (uji orde besaran)" if n_ok == n_tot else f"Didukung sebagian ({n_ok}/{n_tot})")

# 3. JMB-02
j = rs.loc["JMB-02"]
tj = set(tkn[tkn.kode == "JMB-02"].token)
cth = [t for t in ("0xDEADBEEF", "ERR_COMM_TIMEOUT", "#VALUE!") if t in tj]
n9999 = int(sq[(sq.kode == "JMB-02") & (sq.nilai == -9999)].n.sum())
tambah_klaim("JMB-02: teks error, -9999, timestamp 1970, ~0,5% VALID",
             f"{len(tj)} jenis teks galat (termasuk {', '.join(cth)}); {rp(n9999)} sel bernilai -9999; timestamp tertua {j.ts_min_mentah:%Y-%m-%d} ({int(j.n_ts_luar_jendela)} baris di luar periode); VALID {j.persen_valid:.2f}%",
             "Terkonfirmasi" if (j.persen_valid < 1 and j.ts_min_mentah.year == 1970 and n9999 > 0 and len(cth) == 3) else "Sebagian terkonfirmasi")

# 4. PLB-01
p = rs.loc["PLB-01"]
tambah_klaim("PLB-01: ~36 rb baris (duplikat per jam)",
             f"{rp(p.n_baris)} baris; {rp(p.n_baris_lebih_per_jam)} baris lebih pada {rp(p.n_jam_duplikat)} jam; {rp(p.n_durasi_subjam)} baris berjendela {p.durasi_subjam_menit} menit "
             f"dalam {len(blok_plb)} blok: " + ", ".join(f"{a:%d %b %Y}–{b:%d %b %Y}" for a, b in zip(blok_plb.awal, blok_plb.akhir)) +
             f"; {rp(p.n_jam_duplikat_konflik)} jam berisi PM2.5 berbeda-beda",
             "Terkonfirmasi, dengan koreksi penyebab: bacaan sub-jam (:15/:30/:45) pada Sep–Okt, bukan salinan baris")

# 5. MDN-01
md_ = rs.loc["MDN-01"]
tambah_klaim("MDN-01: ~3.600 jam hilang", f"{rp(md_.n_jam_hilang)} jam hilang ({md_.persen_jam_hilang:.1f}%); {rp(md_.n_baris)} baris; gap terpanjang {int(md_.gap_terpanjang_jam)} jam",
             "Terkonfirmasi" if 3500 <= md_.n_jam_hilang <= 3700 else "Berbeda dari klaim")

# 6. MER-01
mer = rs.loc["MER-01"]
n_neg = int(kb_all[(kb_all.kode == "MER-01") & (kb_all.kolom == "pm25_ug_m3")].n_negatif.iloc[0])
tmax = float(dd[(dd.kode == "MER-01") & (dd.kolom == "temperature_c")].maks.iloc[0])
n_t = int(kb_all[(kb_all.kode == "MER-01") & (kb_all.kolom == "temperature_c")].n_luar_rentang.iloc[0])
tambah_klaim("MER-01: ~20% VALID, PM negatif, suhu 99", f"VALID {mer.persen_valid:.2f}%; {rp(n_neg)} baris PM2.5 negatif; {rp(n_t)} baris suhu di luar 5–45 °C (maks {tmax:g} °C); "
             f"semua baris mustahil berstatus SUSPECT (VALID-tapi-mustahil = {int(mer.n_valid_tapi_mustahil)})",
             "Terkonfirmasi" if (18 <= mer.persen_valid <= 22 and n_neg > 0 and tmax >= 99) else "Sebagian terkonfirmasi")

# 7. SM2-01
s2 = sq[(sq.kode == "SM2-01") & (sq.nilai == 999)]
bln_sm2 = P.bulan_nilai_sentinel("SM2-01", 999.0)
nama_bln = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "Mei", 6: "Jun", 7: "Jul", 8: "Agu", 9: "Sep", 10: "Okt", 11: "Nov", 12: "Des"}
bln_txt = ", ".join(f"{nama_bln[k]} ({rp(v)})" for k, v in bln_sm2.items())
tambah_klaim("SM2-01: nilai 999 = saturasi alat (bertanda SUSPECT)",
             f"{rp(s2.n.sum())} sel bernilai 999 pada kolom {', '.join(sorted(s2.kolom.unique()))}; status qc: {', '.join(sorted(s2.qc_flag.unique()))}; "
             f"baris 999 per bulan kalender: {bln_txt} (musiman, hanya pada musim kemarau)",
             "Terkonfirmasi" if set(s2.qc_flag) == {"SUSPECT"} else "Berbeda dari klaim")

# 8. Timestamp
bs = rs[rs.n_ts_bergeser > 0]
tambah_klaim("Timestamp bergeser (mis. 23:48, 00:11, milidetik)",
             f"{len(bs)} dari 27 stasiun punya timestamp bergeser; geseran bertanda terkecil {bs.geser_min_menit.min():.2f} mnt, terbesar {bs.geser_maks_menit.max():.2f} mnt; "
             f"offset konstan terbesar: " + ", ".join(f"{k} {v:+.2f} mnt" for k, v in bs[bs.geser_min_menit == bs.geser_maks_menit].assign(a=lambda d: d.geser_median_menit.abs()).sort_values("a", ascending=False).head(3).geser_median_menit.items()) +
             f"; {len(ms_st)} dari 27 file memakai format dengan pecahan detik (.000); {rp(rs.n_geser_tepat_30menit.sum())} baris tepat :30 (pembulatan ambigu)",
             "Terkonfirmasi (geseran positif dan negatif; format milidetik)")

# 9. qc_flag
sem = sorted(qc.qc_flag.unique())
tambah_klaim("qc_flag: VALID, SUSPECT, CALIBRATING, WARMUP, MAINTENANCE, MISSING, dll.", f"{len(sem)} label ditemukan: {', '.join(sem)}", "Terkonfirmasi (+ label galat alat di JMB-02)")

# 10–11. anomali
bj = musiman[musiman.kode == "BJB-01"].iloc[0]
rek = musiman[musiman.kode != "BJB-01"]
tambah_klaim("BJB-01: pola musim terbalik vs stasiun gambut lain",
             f"Korelasi profil musiman BJB-01 = {bj.korelasi_musiman_vs_rekan:.2f} (stasiun gambut lain {rek.korelasi_musiman_vs_rekan.min():.2f}–{rek.korelasi_musiman_vs_rekan.max():.2f}); "
             f"puncak BJB-01 pada bulan ke-{int(bj.bulan_puncak)} vs bulan {', '.join(map(str, sorted(rek.bulan_puncak.unique())))} pada stasiun lain",
             "Terkonfirmasi (menyimpang kuat)" if bj.korelasi_musiman_vs_rekan < 0.5 else "Tidak terkonfirmasi")
sl = silang.set_index("kode")
tambah_klaim("BJB-01 & SM2-01: PM2.5 tinggi tapi kasus RS rendah",
             "; ".join(f"{k}: peringkat PM2.5 {int(sl.loc[k, 'peringkat_pm25'])} vs peringkat kasus `resp` {int(sl.loc[k, 'peringkat_resp'])} (resp/100 TT = {sl.loc[k, 'resp_per_100_tt']:.1f})" for k in ("BJB-01", "SM2-01")),
             "Terkonfirmasi" if all(sl.loc[k, "selisih_peringkat"] >= 10 for k in ("BJB-01", "SM2-01")) else "Tidak terkonfirmasi")

# 12–13. relasi & cakupan
rr = rel.set_index("kode")
tambah_klaim("Relasi RS-XXX-NN ↔ SENS-0NN ↔ LOC di sites.csv",
             f"R20–R23: {', '.join(f'{k}={rr.loc[k, 'status']}' for k in ('R09', 'R20', 'R21', 'R22', 'R23'))}", "Terkonfirmasi" if all(rr.loc[k, "status"] == "OK" for k in ("R09", "R20", "R21")) else "Ada ketidakcocokan")
tambah_klaim("27 stasiun, 27 RS, 24.432 jam, 2024-01-01…2026-10-14",
             f"{len(ring)} stasiun; {kes['ringkasan']['n_rs']} RS; stasiun dengan tepat {rp(cfg.EXPECTED_HOURS)} baris: {int((ring.n_baris == cfg.EXPECTED_HOURS).sum())}/27",
             "Terkonfirmasi")
klaim = pd.DataFrame(klaim)
pd.set_option("display.max_colwidth", 400)
display(klaim)
pd.set_option("display.max_colwidth", 120)

,klaim,bukti_terukur,status
0,"BJB-01, MDN-01, PSW-01 memakai kolom *_ppm","Stasiun berkolom *_ppm: BJB-01, MDN-01, PSW-01; sites.pm25_unit konsisten (cek R16/R17)",Terkonfirmasi
1,"Konversi PM ×1000 dan gas ppm×BM/24,45×1000",18/18 kombinasi stasiun×polutan masuk rentang kewajaran orde besaran terhadap acuan,Didukung (uji orde besaran)
2,"JMB-02: teks error, -9999, timestamp 1970, ~0,5% VALID","12 jenis teks galat (termasuk 0xDEADBEEF, ERR_COMM_TIMEOUT, #VALUE!); 5.535 sel bernilai -9999; timestamp tertua 1970-01-01 (132 baris di luar periode); VALID 0.50%",Terkonfirmasi
3,PLB-01: ~36 rb baris (duplikat per jam),"36.381 baris; 11.949 baris lebih pada 3.986 jam; 15.933 baris berjendela 15 menit dalam 3 blok: 01 Sep 2024–31 Oct 2024, 01 Sep 2025–31 Oct 2025, 01 Sep 2026–14 Oct 2026; 3.956 jam berisi PM2.5 berbeda-beda","Terkonfirmasi, dengan koreksi penyebab: bacaan sub-jam (:15/:30/:45) pada Sep–Okt, bukan salinan baris"
4,MDN-01: ~3.600 jam hilang,3.632 jam hilang (14.9%); 20.800 baris; gap terpanjang 121 jam,Terkonfirmasi
5,"MER-01: ~20% VALID, PM negatif, suhu 99",VALID 20.02%; 1.780 baris PM2.5 negatif; 749 baris suhu di luar 5–45 °C (maks 99 °C); semua baris mustahil berstatus SUSPECT (VALID-tapi-mustahil = 0),Terkonfirmasi
6,SM2-01: nilai 999 = saturasi alat (bertanda SUSPECT),"2.078 sel bernilai 999 pada kolom pm10_ug_m3, pm25_ug_m3; status qc: SUSPECT; baris 999 per bulan kalender: Jul (285), Agu (267), Sep (270), Okt (217) (musiman, hanya pada musim kemarau)",Terkonfirmasi
7,"Timestamp bergeser (mis. 23:48, 00:11, milidetik)","18 dari 27 stasiun punya timestamp bergeser; geseran bertanda terkecil -15.00 mnt, terbesar 30.00 mnt; offset konstan terbesar: BTM-01 -11.75 mnt, SRG-01 +11.38 mnt, JMB-02 +5.20 mnt; 18 dari 27 file memakai format dengan pecahan detik (.000); 3.983 baris tepat :30 (pembulatan ambigu)",Terkonfirmasi (geseran positif dan negatif; format milidetik)
8,"qc_flag: VALID, SUSPECT, CALIBRATING, WARMUP, MAINTENANCE, MISSING, dll.","12 label ditemukan: CALIBRATING, COMM_LOST, CORRUPT, FATAL_ERR, GARBAGE, MAINTENANCE, MISSING, PARSER_FAIL, SENSOR_DEAD, SUSPECT, VALID, WARMUP",Terkonfirmasi (+ label galat alat di JMB-02)
9,BJB-01: pola musim terbalik vs stasiun gambut lain,"Korelasi profil musiman BJB-01 = 0.26 (stasiun gambut lain 0.86–0.98); puncak BJB-01 pada bulan ke-12 vs bulan 9, 10 pada stasiun lain",Terkonfirmasi (menyimpang kuat)


## 7. Profil data kesehatan (`HLT-01_health_outcomes_daily.csv`)

In [20]:
kr = kes["ringkasan"]
ringkas_kes = pd.Series({
    "Jumlah baris": rp(kr["n_baris"]), "Jumlah RS": kr["n_rs"],
    "Rentang tanggal": f"{kr['tanggal_awal']:%Y-%m-%d} s.d. {kr['tanggal_akhir']:%Y-%m-%d} ({kr['n_hari']} hari)",
    "Baris diharapkan (RS × hari)": rp(kr["baris_diharapkan"]), "Kombinasi tanggal × RS tidak ada": kr["baris_hilang"],
    "…di antaranya dalam periode AQMS (2024-01-01+)": kr["hari_rs_hilang_dalam_periode_aqms"],
    "Baris bertanggal sebelum 2024-01-01": kr["baris_sebelum_periode_aqms"],
    "Duplikat (tanggal, RS)": kr["n_baris_kunci_duplikat"], "Baris identik": kr["n_baris_identik_duplikat"],
    "`dow` tidak cocok dengan tanggal": kr["dow_tidak_cocok"], "`time` ≠ nomor urut per RS": kr["time_tidak_cocok"],
    "Titik embun > suhu": kr["dptp_lebih_besar_dari_suhu"], "RH di luar 0–100": kr["rh_di_luar_0_100"], "Suhu di luar 5–45 °C": kr["suhu_di_luar_5_45"],
    "`resp` < balita + lansia": kr["resp_lebih_kecil_dari_balita_plus_lansia"], "RS dengan kapasitas kosong": kr["kapasitas_kosong"],
}, name="nilai")
display(ringkas_kes.to_frame())
print("Tanggal pada kombinasi yang hilang:", kes["hilang"].tanggal.dt.strftime("%Y-%m-%d").value_counts().to_dict())
print("RS yang punya baris 2023-12-31:", sorted(kes["raw"][kes["raw"].date == "2023-12-31"].hospital_id))

,nilai
Jumlah baris,27.491
Jumlah RS,27
Rentang tanggal,2023-12-31 s.d. 2026-10-14 (1019 hari)
Baris diharapkan (RS × hari),27.513
Kombinasi tanggal × RS tidak ada,22
…di antaranya dalam periode AQMS (2024-01-01+),0
Baris bertanggal sebelum 2024-01-01,5
"Duplikat (tanggal, RS)",0
Baris identik,0
`dow` tidak cocok dengan tanggal,0


Tanggal pada kombinasi yang hilang: {'2023-12-31': 22}
RS yang punya baris 2023-12-31: ['RS-XXX-12', 'RS-XXX-13', 'RS-XXX-16', 'RS-XXX-19', 'RS-XXX-20']


In [21]:
print("Distribusi tiap kolom kasus & cuaca (missing, teks, nol, kuantil; dispersi = varians / rata-rata, >1 berarti over-dispersion):")
display(kes["kolom"].round(2))
print("\nCakupan per RS:")
display(kes["per_rs"].round(1))

Distribusi tiap kolom kasus & cuaca (missing, teks, nol, kuantil; dispersi = varians / rata-rata, >1 berarti over-dispersion):


,kolom,n_kosong,n_teks,n_negatif,n_nol,persen_nol,n_non_bulat,rata2,sd,min,p50,p95,p99,maks,dispersi_var_per_rata2
0,resp_admissions_under5,0,0,0,85,0.3,0.0,6.77,5.41,0.0,5.0,18.0,25.0,114.0,4.32
1,resp_admissions_over60,0,0,0,55,0.2,0.0,8.19,6.32,0.0,6.0,21.0,29.0,127.0,4.88
2,asthma_emergency_cases,0,0,0,829,3.0,0.0,4.83,3.80,0.0,4.0,12.0,18.0,64.0,2.98
3,copd_admissions,0,0,0,1337,4.9,0.0,3.98,3.19,0.0,3.0,10.0,16.0,76.0,2.55
4,resp,0,0,0,13,0.0,0.0,26.67,17.81,0.0,21.0,61.0,77.0,307.0,11.89
5,cvd,0,0,0,63,0.2,0.0,7.52,3.86,0.0,7.0,13.0,17.0,61.0,1.98
6,death,0,0,0,2825,10.3,0.0,1.99,1.58,0.0,2.0,5.0,7.0,24.0,1.26
7,temp_c,0,0,0,0,0.0,NaN,26.33,1.39,22.0,25.9,29.0,29.2,32.1,NaN
8,dptp_c,0,0,0,0,0.0,NaN,22.34,1.24,18.1,21.6,24.9,25.3,29.3,NaN
9,rhum_pct,0,0,0,0,0.0,NaN,79.76,3.31,64.9,78.6,87.1,87.8,99.0,NaN



Cakupan per RS:


,rs,n_baris,hari_unik,awal,akhir,hari_hilang,nama_rumah_sakit,provinsi,kelas_faskes,kapasitas_tempat_tidur
0,RS-XXX-01,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Gambir,DKI Jakarta,Tipe B,300.0
1,RS-XXX-02,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Cilincing,DKI Jakarta,Tipe B,250.0
2,RS-XXX-03,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Cengkareng,DKI Jakarta,Tipe B,280.0
3,RS-XXX-04,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Jekan Raya,Kalimantan Tengah,Tipe B,220.0
4,RS-XXX-05,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Mentawa Baru,Kalimantan Tengah,Tipe C,150.0
5,RS-XXX-06,1018,1018,2024-01-01,2026-10-14,1,RS Paru XXX Cisarua,Jawa Barat,Tipe A (Khusus Paru),200.0
6,RS-XXX-07,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Bogor Tengah,Jawa Barat,Tipe B,260.0
7,RS-XXX-08,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Sleman,D.I. Yogyakarta,Tipe B,250.0
8,RS-XXX-09,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Indrapuri,Aceh,Tipe C,120.0
9,RS-XXX-10,1018,1018,2024-01-01,2026-10-14,1,RSUD XXX Agam,Sumatera Barat,Tipe B,180.0


In [22]:
fig = P.fig_kesehatan_distribusi(kes)
style.simpan_html(fig, "02_kesehatan_distribusi")
fig.show()

fig = P.fig_kesehatan_kelengkapan(kes)
fig.update_layout(title_text="Kelengkapan hari data kesehatan per RS × bulan (% hari)")
style.simpan_html(fig, "02_kesehatan_kelengkapan")
fig.show()

## 8. Daftar masalah kualitas data
Tingkat keparahan: **Kritis** ≥20 % baris terdampak · **Tinggi** ≥5 % · **Sedang** ≥1 % · **Rendah** <1 %. Beberapa jenis memakai penetapan khusus karena dapat diperbaiki
secara deterministik (satuan ppm = Tinggi, timestamp bergeser seragam = Sedang, nilai ekstrem = Sedang, anomali lintas-sumber = Tinggi).
Kolom `tindakan_usulan` hanyalah usulan untuk tahap pembersihan; **belum dijalankan**.

In [23]:
iss = P.susun_masalah(prof, rel, kes, ppm, m["sensors"], curiga=cur)
iss.to_csv(cfg.REPORTS_DIR / "data_quality_issues.csv", index=False, encoding="utf-8-sig")
print(f"{len(iss)} masalah -> reports/data_quality_issues.csv")
display(iss.tingkat.value_counts().sort_index().to_frame("jumlah_masalah").T)
print("\nJumlah masalah per kategori × tingkat:")
display(pd.crosstab(iss.kategori, iss.tingkat).reindex(columns=style.KEPARAHAN_URUTAN, fill_value=0).assign(Total=lambda d: d.sum(axis=1)).sort_values("Total", ascending=False))

75 masalah -> reports/data_quality_issues.csv


tingkat,Kritis,Tinggi,Sedang,Rendah
jumlah_masalah,15,15,38,7



Jumlah masalah per kategori × tingkat:


tingkat,Kritis,Tinggi,Sedang,Rendah,Total
kategori,,,,,
Timestamp bergeser dari awal jam,1,0,17,0,18
Baris tidak berstatus VALID,4,1,7,0,12
Nilai ekstrem (perlu verifikasi),0,0,6,0,6
Nilai kosong (missing),2,1,2,0,5
Label qc_flag di luar spesifikasi metadata,1,3,0,0,4
Status sensor tidak aktif penuh,0,0,4,0,4
Anomali pola (perlu investigasi),0,4,0,0,4
Satuan tidak standar (ppm),0,3,0,0,3
Jam hilang (tidak ada baris),0,1,0,1,2


In [24]:
pd.set_option("display.max_colwidth", 140)
cols = ["id_masalah", "entitas", "kategori", "kolom", "deskripsi", "jumlah_terdampak", "persen_terdampak", "tingkat"]
print("Masalah Kritis dan Tinggi:")
display(iss[iss.tingkat.isin(["Kritis", "Tinggi"])][cols].reset_index(drop=True))
pd.set_option("display.max_colwidth", 120)

Masalah Kritis dan Tinggi:


,id_masalah,entitas,kategori,kolom,deskripsi,jumlah_terdampak,persen_terdampak,tingkat
0,DQ-001,JMB-02,Baris tidak berstatus VALID,qc_flag,Hanya 0.5% VALID; 8 label qc_flag berbeda.,24310,99.50,Kritis
1,DQ-002,JMB-02,Label qc_flag di luar spesifikasi metadata,qc_flag,"Label selain VALID/SUSPECT/CALIBRATING/MISSING (mis. WARMUP, MAINTENANCE, kode galat).",20963,85.80,Kritis
2,DQ-003,JMB-02,Nilai di luar rentang fisik (cuaca),"temperature_c,humidity_pct,wind_speed_ms,wind_direction_deg","Suhu <5/>45 °C, RH di luar 0–100, angin >40 m/s, arah di luar 0–360. Rincian: temperature_c=11862, humidity_pct=18255, wind_speed_ms=204...",20498,83.90,Kritis
3,DQ-004,MER-01,Baris tidak berstatus VALID,qc_flag,Hanya 20.0% VALID; 6 label qc_flag berbeda.,19541,79.98,Kritis
4,DQ-005,JMB-02,Nilai sentinel (999/-9999 dst.),"pm25_ug_m3,pm10_ug_m3,so2_ug_m3,co_ug_m3,o3_ug_m3,no2_ug_m3,temperature_c,wind_direction_deg","Nilai pengganti galat/saturasi alat. Rincian: pm25_ug_m3=891, pm10_ug_m3=889, so2_ug_m3=976, co_ug_m3=7571, o3_ug_m3=5835, no2_ug_m3=967...",15597,63.84,Kritis
5,DQ-006,PLB-01,Durasi jendela ≠ 1 jam,timestamp_end,"Selisih timestamp_end − timestamp_start bukan 3600 detik. 15933 baris berjendela 15 menit, terkumpul dalam 3 blok jam berurutan (2024-09...",15933,43.79,Kritis
6,DQ-007,MER-01,Nilai kosong (missing),semua kolom numerik,Baris dengan ≥1 kolom numerik kosong.,9778,40.02,Kritis
7,DQ-008,JMB-02,Nilai non-numerik (teks galat),semua kolom numerik,"12 jenis teks: 0xDEADBEEF×5706, ERR_COMM_TIMEOUT×5562, BAD_CHECKSUM×5108, VOLT_SURGE_99V×5072, #VALUE!×5042, ADC_SAT_FAIL×5041….",8997,36.82,Kritis
8,DQ-009,PLB-01,Timestamp bergeser dari awal jam,timestamp_start,Geseran bertanda median 15.0 mnt (rentang -15.0…30.0 mnt); sebaran offset dari awal jam: +15 mnt: 3983; +30 mnt: 3983; +45 mnt: 3983. 39...,11949,32.84,Kritis
9,DQ-010,PLB-01,Duplikat per jam,timestamp_start,"11949 baris lebih; 3986 jam berisi >1 baris, 3956 di antaranya bernilai PM2.5 berbeda. Penyebab dominan: bacaan sub-jam (:15/:30/:45), b...",11949,32.84,Kritis


In [25]:
print("Peta ringkasan masalah: stasiun × kategori (warna = tingkat keparahan tertinggi)")
fig = P.fig_ringkasan_masalah(iss, urutan)
style.simpan_html(fig, "02_ringkasan_masalah")
fig.show()

print("\nTabel ringkasan per stasiun:")
per_st = (iss[iss.entitas.isin(urutan)].assign(t=lambda d: d.tingkat.astype(str))
          .pivot_table(index="entitas", columns="t", values="id_masalah", aggfunc="count", fill_value=0)
          .reindex(index=urutan, columns=style.KEPARAHAN_URUTAN, fill_value=0))
per_st["Total"] = per_st.sum(axis=1)
display(per_st.style.background_gradient(cmap="Reds", subset=["Kritis", "Tinggi"], axis=None))

Peta ringkasan masalah: stasiun × kategori (warna = tingkat keparahan tertinggi)



Tabel ringkasan per stasiun:


t,Kritis,Tinggi,Sedang,Rendah,Total
entitas,,,,,
BGL-01,0,0,1,0,1
MDN-01,1,4,3,0,8
PSW-01,0,1,1,0,2
BTM-01,0,0,1,0,1
JMB-01,0,0,1,0,1
JMB-02,8,0,3,5,16
PKU-01,0,0,1,0,1
PLB-01,4,1,1,0,6
IDP-01,0,0,0,0,0


## 9. Menyimpan tabel profil
Tabel keluaran disimpan agar notebook/laporan/dashboard berikutnya membaca angka dari sumber yang sama.

In [26]:
paket = {
    "inventaris_file": inv, "profil_stasiun": ring, "profil_kolom": prof["kolom"], "profil_token": prof["token"], "profil_qc": prof["qc"],
    "profil_sentinel_qc": prof["sentinel_qc"], "statistik_mentah": prof["desk"], "ketersediaan_bulanan": prof["bulanan"],
    "relasi_pemeriksaan": rel, "pasangan_stasiun_rs": pas, "verifikasi_ppm": ppm, "tanda_curiga": cur, "musiman_gambut": musiman,
    "silang_pm25_kasus": silang, "verifikasi_klaim": klaim, "kesehatan_kolom": kes["kolom"], "kesehatan_per_rs": kes["per_rs"],
}
for nama, df in paket.items():
    df.to_parquet(PROF_DIR / f"{nama}.parquet", index=False)
pd.DataFrame([kes["ringkasan"]]).to_parquet(PROF_DIR / "kesehatan_ringkasan.parquet", index=False)
print(f"{len(paket) + 1} tabel -> {PROF_DIR.relative_to(ROOT)}")
print("Pastikan data/raw tidak berubah: jumlah file =", len(list(cfg.RAW_DIR.rglob('*.csv'))))

18 tabel -> data\processed\profiling
Pastikan data/raw tidak berubah: jumlah file = 33


## 10. Temuan
Sel kode berikut menyusun teks temuan **langsung dari tabel di atas** (agar angka identik dengan keluaran notebook), menyimpannya ke
`reports/findings/02_data_understanding.md`, lalu menampilkannya. Teks yang sama ditempel sebagai sel markdown terakhir notebook ini.

In [27]:
def md_tabel(df):
    return df.to_markdown(index=False)

R = lambda k, c: ring.set_index("kode").loc[k, c]
n_aqms = int((inv.tabel == "aqms").sum())
kritis_st = sorted(set(iss[(iss.tingkat == "Kritis") & iss.entitas.isin(ring.kode)].entitas))
tinggi_st = sorted(set(iss[(iss.tingkat == "Tinggi") & iss.entitas.isin(ring.kode)].entitas) - set(kritis_st))
bersih = sorted(set(cur[~cur.mencurigakan].kode))
n_sh = ring[(ring.n_ts_bergeser > 0)]
cnt = iss.tingkat.value_counts()
jenis_jmb = tok[tok.kode == "JMB-02"].assign(j=lambda d: d.token.map(P.jenis_token)).groupby("j").n.sum().sort_values(ascending=False)
pm_ppm = ppm[ppm.polutan == "pm25"].set_index("kode")
mus = musiman.set_index("kode")
n_konstan = int((n_sh.geser_min_menit == n_sh.geser_maks_menit).sum())
nonvalid_bersih = (100 - ring.set_index("kode").loc[bersih, "persen_valid"])
st_jam_hilang = ", ".join(ring[ring.n_jam_hilang > 0].kode)
n_valid_mustahil = int(ring.n_valid_tapi_mustahil.sum())
ekst = ring.set_index("kode").loc[["KTG-01", "KTG-02", "PKN-01"], ["n_baris_ekstrem", "n_baris_ekstrem_valid"]]
blok_txt = "; ".join(f"{a:%d %b %Y}–{b:%d %b %Y}" for a, b in zip(blok_plb.awal, blok_plb.akhir))
bln_sm2_txt = ", ".join(nama_bln[k] for k in bln_sm2.index)
pm_max = lambda k: dd[(dd.kode == k) & (dd.kanon == "pm25")].maks.iloc[0]

temuan = f"""# 02 — Data Understanding: Temuan
> Dihasilkan otomatis oleh `notebooks/02_data_understanding.ipynb`. Semua angka berasal dari `data/processed/profiling/*.parquet`,
> `reports/data_quality_issues.csv`, dan `reports/data_dictionary.csv`. **Tidak ada data mentah yang diubah atau dibersihkan** pada tahap ini.

## 1. Cakupan dan struktur data
- **{len(inv)} file** dibaca: {n_aqms} file AQMS ({rp(ring.n_baris.sum())} baris), 1 file kesehatan ({rp(kr["n_baris"])} baris), 3 tabel master, 2 metadata.
- Periode AQMS: {cfg.WINDOW_START:%Y-%m-%d} s.d. {cfg.WINDOW_END:%Y-%m-%d} = **{rp(cfg.EXPECTED_HOURS)} jam** per stasiun. Hanya **{int((ring.n_baris == cfg.EXPECTED_HOURS).sum())} dari {len(ring)}** file memuat tepat jumlah baris itu.
- Kunci antar-tabel **cocok sepenuhnya**: {int((rel[rel.kode != "R27"].status == "OK").sum())} pemeriksaan kunci/konsistensi OK. Satu-satunya ketidakcocokan adalah jumlah baris file (R27: {rel.set_index("kode").loc["R27", "detail"]}), ditambah {int((rel.status == "INFO").sum())} butir informasi (stasiun ppm, status sensor). Pasangan stasiun ↔ sensor ↔ lokasi ↔ RS (`SENS-0NN ↔ RS-XXX-NN`) lengkap 27/27 dan provinsi/kota/kecamatan RS sama dengan stasiun pasangannya (pencocokan longgar).
- Sebaran stasiun: Sumatera {int((pas.pulau == "Sumatera").sum())}, Jawa {int((pas.pulau == "Jawa").sum())}, Kalimantan {int((pas.pulau == "Kalimantan").sum())}, Sulawesi {int((pas.pulau == "Sulawesi").sum())}, Papua {int((pas.pulau == "Papua").sum())}; kelompok lokasi: """ + ", ".join(f"{k} {int((pas.kelompok_lokasi == k).sum())}" for k in cfg.URUTAN_KELOMPOK) + f""".
- Status sensor: {int((m["sensors"].status_operasional == "Active").sum())} Active, {int((m["sensors"].status_operasional == "Intermittent").sum())} Intermittent (PLB-01, MER-01, MDN-01), {int((m["sensors"].status_operasional == "Defective").sum())} Defective (JMB-02). Status ini sejalan dengan stasiun terparah yang terukur (JMB-02, MER-01, MDN-01, PLB-01); SM2-01 berstatus Active meski hanya {R("SM2-01", "persen_valid"):.1f}% VALID.

## 2. Hasil verifikasi masalah yang sudah diketahui
{md_tabel(klaim)}

## 3. Temuan utama kualitas data AQMS
1. **Satuan tidak seragam.** {", ".join(ppm_st)} memakai kolom `*_ppm` untuk **semua** polutan, termasuk PM. Median PM2.5 setelah ×1000 berada pada {pm_ppm.median_terkonversi_ugm3.min():.1f}–{pm_ppm.median_terkonversi_ugm3.max():.1f} µg/m³, sangat dekat dengan `baseline_pm25` stasiunnya ({pm_ppm.acuan_ugm3.min():.1f}–{pm_ppm.acuan_ugm3.max():.1f} µg/m³); tanpa konversi, nilainya ±1000× lebih kecil. Hipotesis konversi didukung pada {int(ppm.hipotesis_didukung.sum())}/{len(ppm)} kombinasi (uji orde besaran).
2. **Satu stasiun tidak layak pakai: JMB-02 (Muaro Jambi).** Hanya {R("JMB-02", "persen_valid"):.2f}% baris `VALID`; {rp(R("JMB-02", "n_baris_teks"))} baris ({R("JMB-02", "n_baris_teks") / R("JMB-02", "n_baris") * 100:.1f}%) memuat teks galat dari {len(jenis_jmb)} kelompok jenis ({", ".join(f"{k} ({rp(v)} sel)" for k, v in jenis_jmb.items())}); {rp(R("JMB-02", "n_baris_mustahil"))} baris ({R("JMB-02", "n_baris_mustahil") / R("JMB-02", "n_baris") * 100:.1f}%) memuat nilai mustahil; {rp(R("JMB-02", "n_ts_luar_jendela"))} baris bertimestamp 1970; PM2.5 > PM10 pada {rp(R("JMB-02", "n_pm25_gt_pm10"))} baris. Status sensor `Defective`. Perlakuan awal: kecualikan dari analisis konsentrasi dan kesehatan, tetapi **pertahankan sebagai bukti** untuk Tujuan 5 (kecukupan jaringan).
3. **Stasiun dengan cakupan VALID rendah.** MER-01: {R("MER-01", "persen_valid"):.1f}% VALID ({rp(R("MER-01", "n_baris_kosong"))} baris berisi sel kosong, {rp(R("MER-01", "n_baris_mustahil"))} baris mustahil, semuanya `SUSPECT`). MDN-01: {rp(R("MDN-01", "n_jam_hilang"))} jam tidak ada baris ({R("MDN-01", "persen_jam_hilang"):.1f}%), gap terpanjang {int(R("MDN-01", "gap_terpanjang_jam"))} jam, jam VALID hanya {R("MDN-01", "persen_jam_valid"):.1f}% dari 24.432. SM2-01: {R("SM2-01", "persen_valid"):.1f}% VALID; {rp(R("SM2-01", "n_baris_mustahil"))} baris bernilai 999 pada PM2.5/PM10, seluruhnya `SUSPECT` dan hanya pada bulan {bln_sm2_txt} (musim kemarau), konsisten dengan dugaan saturasi alat saat asap pekat. Pengecualian `SUSPECT` secara membabi buta berisiko **menghapus episode asap puncak** pada SM2-01.
4. **PLB-01: baris berlebih berasal dari logger yang merekam tiap 15 menit pada musim asap.** {rp(R("PLB-01", "n_baris"))} baris vs 24.432 jam. {rp(R("PLB-01", "n_durasi_subjam"))} baris berjendela {R("PLB-01", "durasi_subjam_menit")} menit, terkumpul dalam {len(blok_plb)} blok jam berurutan ({blok_txt}), yaitu **hanya pada September–Oktober** (4 bacaan/jam pada :00/:15/:30/:45); di luar blok itu data per jam normal. Ini bukan duplikat murni: aturan "hapus duplikat" akan membuang data sah, sedangkan pembulatan ke jam terdekat memindahkan baris :45 ke jam berikutnya dan menciptakan duplikat semu tepat pada periode asap ({rp(R("PLB-01", "n_jam_duplikat"))} jam berisi >1 baris, {rp(R("PLB-01", "n_jam_duplikat_konflik"))} jam di antaranya bernilai PM2.5 berbeda). Gunakan agregasi per jam penuh (floor). Selain itu terdapat {rp(R("PLB-01", "n_baris_kosong"))} baris `MISSING` (kosong).
5. **Timestamp.** {len(n_sh)} dari {len(ring)} stasiun memiliki timestamp bergeser dari awal jam (geseran bertanda {n_sh.geser_min_menit.min():.1f} s.d. +{n_sh.geser_maks_menit.max():.1f} menit; {n_konstan} stasiun memakai satu offset konstan, sisanya bervariasi). {rp(ring.n_geser_tepat_30menit.sum())} baris tepat :30 sehingga pembulatan ke "jam terdekat" ambigu (terjadi pada PLB-01 yang sub-jam). Seluruh `timestamp_start` dapat di-parse ({int(ring.ts_tak_terbaca.sum())} baris tak terbaca); satu-satunya timestamp di luar periode adalah 1970 pada JMB-02.
6. **`qc_flag` konsisten dengan nilai mustahil, tetapi tidak cukup sebagai satu-satunya filter.** Ditemukan {len(sem)} label (metadata hanya mendefinisikan 4). Menurut aturan rentang fisik, ada {n_valid_mustahil} baris `VALID` bernilai mustahil; nilai mustahil di MER-01, SM2-01, dan JMB-02 semuanya sudah ditandai non-VALID oleh alat. Namun `qc_flag` (a) tidak menandai satuan salah label (ppm), (b) menandai BJB-01 100% `VALID` padahal polanya menyimpang, dan (c) memberi `SUSPECT` pada nilai saturasi SM2-01 yang kemungkinan merupakan episode asap nyata. Catatan: angka `999` pada CO (mis. KTG-01 dan KTG-02, berstatus `VALID`) **bukan** sentinel karena 999 µg/m³ adalah konsentrasi CO yang wajar.
7. **Nilai ekstrem yang mungkin nyata.** PM2.5 maksimum: KTG-01 {rp(pm_max("KTG-01"))}, KTG-02 {rp(pm_max("KTG-02"))}, PKN-01 {rp(pm_max("PKN-01"))} µg/m³. Dari {rp(ekst.n_baris_ekstrem.sum())} baris PM >500 µg/m³ di tiga stasiun itu, {rp(ekst.n_baris_ekstrem_valid.sum())} berstatus `VALID`. Ketiganya berada di kawasan gambut/hutan, jadi nilai ini **tidak boleh dibuang otomatis** sebagai pencilan; verifikasi dengan CO, rasio PM2.5/PM10, dan musim pada EDA.
8. **Anomali yang perlu dijelaskan sebelum analisis hubungan PM2.5–kesehatan.** (a) BJB-01 berpola musiman menyimpang dari stasiun gambut lain (r = {mus.loc["BJB-01", "korelasi_musiman_vs_rekan"]:.2f} vs {mus.drop("BJB-01").korelasi_musiman_vs_rekan.min():.2f}–{mus.drop("BJB-01").korelasi_musiman_vs_rekan.max():.2f}; puncak bulan ke-{int(mus.loc["BJB-01", "bulan_puncak"])}, bukan Sep–Okt) dan 100% `VALID` meski unit ppm. (b) BJB-01 dan SM2-01 berada di peringkat PM2.5 ke-{int(sl.loc["BJB-01", "peringkat_pm25"])} dan ke-{int(sl.loc["SM2-01", "peringkat_pm25"])}, tetapi kasus pernapasan RS pasangannya di peringkat ke-{int(sl.loc["BJB-01", "peringkat_resp"])} dan ke-{int(sl.loc["SM2-01", "peringkat_resp"])} (dari 27). Penjelasan yang mungkin (belum diuji): paparan di stasiun tidak mewakili populasi RS pasangannya, atau hubungan PM2.5–kasus pada data sintetis ini memang lemah di wilayah tersebut. **Tidak boleh disimpulkan sebagai kausalitas.**

## 4. Temuan data kesehatan
- {rp(kr["n_baris"])} baris, {kr["n_rs"]} RS, {kr["tanggal_awal"]:%Y-%m-%d} s.d. {kr["tanggal_akhir"]:%Y-%m-%d}. **Tidak ada duplikat** (tanggal, RS), **tidak ada missing/teks/negatif** pada kolom kasus maupun cuaca, `dow` cocok dengan tanggal, dan kewajaran fisik terpenuhi (titik embun ≤ suhu, RH 0–100).
- {kr["baris_hilang"]} kombinasi RS×hari tidak ada, **seluruhnya 2023-12-31**: hanya {kr["baris_sebelum_periode_aqms"]} RS yang punya baris 2023-12-31 (sebelum AQMS mulai). **Dalam periode AQMS tidak ada hari hilang** ({kr["hari_rs_hilang_dalam_periode_aqms"]}).
- `time` adalah nomor urut hari **per RS** (bukan indeks global): {kr["time_tidak_cocok"]} ketidakcocokan terhadap urutan per RS.
- Semua kolom kasus berupa bilangan bulat dan **over-dispersed** (varians/rata-rata {kes["kolom"].set_index("kolom").loc[P.KOLOM_KASUS, "dispersi_var_per_rata2"].min():.1f}–{kes["kolom"].set_index("kolom").loc[P.KOLOM_KASUS, "dispersi_var_per_rata2"].max():.1f}), sehingga model hubungan harus memakai Poisson dengan dispersi (quasi-Poisson/negatif binomial), bukan OLS.
- `resp` lebih kecil dari balita + lansia pada {kr["resp_lebih_kecil_dari_balita_plus_lansia"]} baris, jadi `resp` jangan diasumsikan mencakup kolom umur. `kapasitas_tempat_tidur` terisi untuk semua {27 - kr["kapasitas_kosong"]} RS (booklet menyebut dapat kosong), tetapi {int((pas.kapasitas_tempat_tidur == 180).sum())} RS bernilai seragam 180 (kemungkinan nilai isian).
- Kasus harian per RS tidak dapat dibakukan terhadap penduduk (tidak ada denominator populasi).

## 5. Ringkasan daftar masalah (`reports/data_quality_issues.csv`)
- Total **{len(iss)} masalah**: {int(cnt.get("Kritis", 0))} Kritis, {int(cnt.get("Tinggi", 0))} Tinggi, {int(cnt.get("Sedang", 0))} Sedang, {int(cnt.get("Rendah", 0))} Rendah.
- Stasiun dengan masalah Kritis: {", ".join(kritis_st)}. Hanya Tinggi: {", ".join(tinggi_st)}.
- Stasiun tanpa tanda kecurigaan pada pemeriksaan awal ({len(bersih)}): {", ".join(bersih)}. Pada stasiun-stasiun ini baris non-`VALID` berkisar {nonvalid_bersih.min():.1f}–{nonvalid_bersih.max():.1f}% (label `SUSPECT`/`CALIBRATING`), semuanya berunit µg/m³ dan sebagian memiliki timestamp bergeser.

## 6. Usulan aturan pembersihan untuk tahap berikutnya (belum dijalankan)
| No | Aturan | Baris/stasiun terdampak |
|---|---|---|
| 1 | Konversi `*_ppm` → µg/m³ (PM ×1000; gas ppm×BM/24,45×1000) dan beri nama kolom seragam | {", ".join(ppm_st)} |
| 2 | Ubah teks galat, string kosong, dan sentinel (kecuali CO=999) menjadi `NaN` | {rp(ring.n_baris_teks.sum())} baris berteks galat; {rp(ring.n_baris_mustahil.sum())} baris bernilai mustahil |
| 3 | Terapkan batas fisik (PM ≥ 0; suhu 5–45 °C; RH 0–100; angin ≤ 40 m/s; arah 0–360°) di luar `qc_flag` | pengaman tambahan; {n_valid_mustahil} baris `VALID` bernilai mustahil pada pemeriksaan ini |
| 4 | Timestamp: agregasi ke jam penuh (floor) untuk stasiun beresolusi sub-jam; bulatkan ke jam terdekat untuk offset konstan; buang timestamp luar periode | {len(n_sh)} stasiun bergeser; PLB-01 sub-jam; JMB-02 1970 |
| 5 | Reindex ke 24.432 jam; jam tanpa data tetap `NaN` (jangan diimputasi untuk PM) | {rp(ring.n_jam_hilang.sum())} jam hilang ({st_jam_hilang}) |
| 6 | Pemetaan `qc_flag`: VALID dipakai; `SUSPECT` dikecualikan dari analisis utama **kecuali** saturasi saat asap (SM2-01) diperlakukan khusus; uji sensitivitas | {len(sem)} label |
| 7 | Kecualikan JMB-02 dari analisis konsentrasi/kesehatan; MER-01 hanya dengan peringatan kelengkapan | 2 stasiun |
| 8 | Sampel kesehatan dibatasi pada 2024-01-01…2026-10-14 | {kr["baris_sebelum_periode_aqms"]} baris |

## 7. Keterbatasan dan hal yang belum diverifikasi
- Faktor ppm → µg/m³ diuji dengan orde besaran terhadap `baseline_pm25` dan median stasiun lain, bukan terhadap pengukuran referensi; PM dalam "ppm" hanyalah hipotesis (mg/m³ yang salah label).
- Kriteria "mencurigakan" (rasio 0,5–2× `baseline_pm25`, PM > 500 µg/m³, selisih peringkat ≥ 10) adalah ambang praktis, bukan standar.
- PM2.5 harian pada pemeriksaan silang adalah pendekatan profiling (≥12 jam VALID/hari), **bukan** data bersih final. Korelasi musiman memakai {len(musiman)} stasiun gambut (JMB-02 dikecualikan) sehingga bersifat deskriptif.
- Tidak ada pengujian kausal; semua perbandingan peringkat/korelasi hanyalah penanda untuk investigasi pada notebook 03–04.
"""
(cfg.FINDINGS_DIR / "02_data_understanding.md").write_text(temuan, encoding="utf-8")
print(f"Tersimpan: reports/findings/02_data_understanding.md ({len(temuan.splitlines())} baris)")
display(Markdown(temuan))

Tersimpan: reports/findings/02_data_understanding.md (68 baris)


# 02 — Data Understanding: Temuan
> Dihasilkan otomatis oleh `notebooks/02_data_understanding.ipynb`. Semua angka berasal dari `data/processed/profiling/*.parquet`,
> `reports/data_quality_issues.csv`, dan `reports/data_dictionary.csv`. **Tidak ada data mentah yang diubah atau dibersihkan** pada tahap ini.

## 1. Cakupan dan struktur data
- **33 file** dibaca: 27 file AQMS (667.981 baris), 1 file kesehatan (27.491 baris), 3 tabel master, 2 metadata.
- Periode AQMS: 2024-01-01 s.d. 2026-10-14 = **24.432 jam** per stasiun. Hanya **25 dari 27** file memuat tepat jumlah baris itu.
- Kunci antar-tabel **cocok sepenuhnya**: 24 pemeriksaan kunci/konsistensi OK. Satu-satunya ketidakcocokan adalah jumlah baris file (R27: MDN-01=20800; PLB-01=36381), ditambah 2 butir informasi (stasiun ppm, status sensor). Pasangan stasiun ↔ sensor ↔ lokasi ↔ RS (`SENS-0NN ↔ RS-XXX-NN`) lengkap 27/27 dan provinsi/kota/kecamatan RS sama dengan stasiun pasangannya (pencocokan longgar).
- Sebaran stasiun: Sumatera 10, Jawa 7, Kalimantan 6, Sulawesi 2, Papua 2; kelompok lokasi: Perkotaan 7, Jalan Raya 2, Industri 4, Gambut 8, Latar Bersih 6.
- Status sensor: 23 Active, 3 Intermittent (PLB-01, MER-01, MDN-01), 1 Defective (JMB-02). Status ini sejalan dengan stasiun terparah yang terukur (JMB-02, MER-01, MDN-01, PLB-01); SM2-01 berstatus Active meski hanya 85.0% VALID.

## 2. Hasil verifikasi masalah yang sudah diketahui
| klaim                                                                    | bukti_terukur                                                                                                                                                                                                                                                                                 | status                                                                                                 |
|:-------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------|
| BJB-01, MDN-01, PSW-01 memakai kolom *_ppm                               | Stasiun berkolom *_ppm: BJB-01, MDN-01, PSW-01; sites.pm25_unit konsisten (cek R16/R17)                                                                                                                                                                                                       | Terkonfirmasi                                                                                          |
| Konversi PM ×1000 dan gas ppm×BM/24,45×1000                              | 18/18 kombinasi stasiun×polutan masuk rentang kewajaran orde besaran terhadap acuan                                                                                                                                                                                                           | Didukung (uji orde besaran)                                                                            |
| JMB-02: teks error, -9999, timestamp 1970, ~0,5% VALID                   | 12 jenis teks galat (termasuk 0xDEADBEEF, ERR_COMM_TIMEOUT, #VALUE!); 5.535 sel bernilai -9999; timestamp tertua 1970-01-01 (132 baris di luar periode); VALID 0.50%                                                                                                                          | Terkonfirmasi                                                                                          |
| PLB-01: ~36 rb baris (duplikat per jam)                                  | 36.381 baris; 11.949 baris lebih pada 3.986 jam; 15.933 baris berjendela 15 menit dalam 3 blok: 01 Sep 2024–31 Oct 2024, 01 Sep 2025–31 Oct 2025, 01 Sep 2026–14 Oct 2026; 3.956 jam berisi PM2.5 berbeda-beda                                                                                | Terkonfirmasi, dengan koreksi penyebab: bacaan sub-jam (:15/:30/:45) pada Sep–Okt, bukan salinan baris |
| MDN-01: ~3.600 jam hilang                                                | 3.632 jam hilang (14.9%); 20.800 baris; gap terpanjang 121 jam                                                                                                                                                                                                                                | Terkonfirmasi                                                                                          |
| MER-01: ~20% VALID, PM negatif, suhu 99                                  | VALID 20.02%; 1.780 baris PM2.5 negatif; 749 baris suhu di luar 5–45 °C (maks 99 °C); semua baris mustahil berstatus SUSPECT (VALID-tapi-mustahil = 0)                                                                                                                                        | Terkonfirmasi                                                                                          |
| SM2-01: nilai 999 = saturasi alat (bertanda SUSPECT)                     | 2.078 sel bernilai 999 pada kolom pm10_ug_m3, pm25_ug_m3; status qc: SUSPECT; baris 999 per bulan kalender: Jul (285), Agu (267), Sep (270), Okt (217) (musiman, hanya pada musim kemarau)                                                                                                    | Terkonfirmasi                                                                                          |
| Timestamp bergeser (mis. 23:48, 00:11, milidetik)                        | 18 dari 27 stasiun punya timestamp bergeser; geseran bertanda terkecil -15.00 mnt, terbesar 30.00 mnt; offset konstan terbesar: BTM-01 -11.75 mnt, SRG-01 +11.38 mnt, JMB-02 +5.20 mnt; 18 dari 27 file memakai format dengan pecahan detik (.000); 3.983 baris tepat :30 (pembulatan ambigu) | Terkonfirmasi (geseran positif dan negatif; format milidetik)                                          |
| qc_flag: VALID, SUSPECT, CALIBRATING, WARMUP, MAINTENANCE, MISSING, dll. | 12 label ditemukan: CALIBRATING, COMM_LOST, CORRUPT, FATAL_ERR, GARBAGE, MAINTENANCE, MISSING, PARSER_FAIL, SENSOR_DEAD, SUSPECT, VALID, WARMUP                                                                                                                                               | Terkonfirmasi (+ label galat alat di JMB-02)                                                           |
| BJB-01: pola musim terbalik vs stasiun gambut lain                       | Korelasi profil musiman BJB-01 = 0.26 (stasiun gambut lain 0.86–0.98); puncak BJB-01 pada bulan ke-12 vs bulan 9, 10 pada stasiun lain                                                                                                                                                        | Terkonfirmasi (menyimpang kuat)                                                                        |
| BJB-01 & SM2-01: PM2.5 tinggi tapi kasus RS rendah                       | BJB-01: peringkat PM2.5 1 vs peringkat kasus `resp` 25 (resp/100 TT = 3.5); SM2-01: peringkat PM2.5 2 vs peringkat kasus `resp` 24 (resp/100 TT = 2.3)                                                                                                                                        | Terkonfirmasi                                                                                          |
| Relasi RS-XXX-NN ↔ SENS-0NN ↔ LOC di sites.csv                           | R20–R23: R09=OK, R20=OK, R21=OK, R22=OK, R23=OK                                                                                                                                                                                                                                               | Terkonfirmasi                                                                                          |
| 27 stasiun, 27 RS, 24.432 jam, 2024-01-01…2026-10-14                     | 27 stasiun; 27 RS; stasiun dengan tepat 24.432 baris: 25/27                                                                                                                                                                                                                                   | Terkonfirmasi                                                                                          |

## 3. Temuan utama kualitas data AQMS
1. **Satuan tidak seragam.** BJB-01, MDN-01, PSW-01 memakai kolom `*_ppm` untuk **semua** polutan, termasuk PM. Median PM2.5 setelah ×1000 berada pada 43.5–46.9 µg/m³, sangat dekat dengan `baseline_pm25` stasiunnya (35.0–54.3 µg/m³); tanpa konversi, nilainya ±1000× lebih kecil. Hipotesis konversi didukung pada 18/18 kombinasi (uji orde besaran).
2. **Satu stasiun tidak layak pakai: JMB-02 (Muaro Jambi).** Hanya 0.50% baris `VALID`; 8.997 baris (36.8%) memuat teks galat dari 7 kelompok jenis (galat perangkat keras sensor (15.988 sel), galat komunikasi/integritas data (10.670 sel), kode heksadesimal (isi memori) (5.706 sel), galat formula spreadsheet (5.042 sel), nilai tak hingga (5.032 sel), byte kosong/null (460 sel), kode galat umum (273 sel)); 24.308 baris (99.5%) memuat nilai mustahil; 132 baris bertimestamp 1970; PM2.5 > PM10 pada 4.897 baris. Status sensor `Defective`. Perlakuan awal: kecualikan dari analisis konsentrasi dan kesehatan, tetapi **pertahankan sebagai bukti** untuk Tujuan 5 (kecukupan jaringan).
3. **Stasiun dengan cakupan VALID rendah.** MER-01: 20.0% VALID (9.778 baris berisi sel kosong, 2.529 baris mustahil, semuanya `SUSPECT`). MDN-01: 3.632 jam tidak ada baris (14.9%), gap terpanjang 121 jam, jam VALID hanya 68.1% dari 24.432. SM2-01: 85.0% VALID; 1.039 baris bernilai 999 pada PM2.5/PM10, seluruhnya `SUSPECT` dan hanya pada bulan Jul, Agu, Sep, Okt (musim kemarau), konsisten dengan dugaan saturasi alat saat asap pekat. Pengecualian `SUSPECT` secara membabi buta berisiko **menghapus episode asap puncak** pada SM2-01.
4. **PLB-01: baris berlebih berasal dari logger yang merekam tiap 15 menit pada musim asap.** 36.381 baris vs 24.432 jam. 15.933 baris berjendela 15 menit, terkumpul dalam 3 blok jam berurutan (01 Sep 2024–31 Oct 2024; 01 Sep 2025–31 Oct 2025; 01 Sep 2026–14 Oct 2026), yaitu **hanya pada September–Oktober** (4 bacaan/jam pada :00/:15/:30/:45); di luar blok itu data per jam normal. Ini bukan duplikat murni: aturan "hapus duplikat" akan membuang data sah, sedangkan pembulatan ke jam terdekat memindahkan baris :45 ke jam berikutnya dan menciptakan duplikat semu tepat pada periode asap (3.986 jam berisi >1 baris, 3.956 jam di antaranya bernilai PM2.5 berbeda). Gunakan agregasi per jam penuh (floor). Selain itu terdapat 5.136 baris `MISSING` (kosong).
5. **Timestamp.** 18 dari 27 stasiun memiliki timestamp bergeser dari awal jam (geseran bertanda -15.0 s.d. +30.0 menit; 17 stasiun memakai satu offset konstan, sisanya bervariasi). 3.983 baris tepat :30 sehingga pembulatan ke "jam terdekat" ambigu (terjadi pada PLB-01 yang sub-jam). Seluruh `timestamp_start` dapat di-parse (0 baris tak terbaca); satu-satunya timestamp di luar periode adalah 1970 pada JMB-02.
6. **`qc_flag` konsisten dengan nilai mustahil, tetapi tidak cukup sebagai satu-satunya filter.** Ditemukan 12 label (metadata hanya mendefinisikan 4). Menurut aturan rentang fisik, ada 0 baris `VALID` bernilai mustahil; nilai mustahil di MER-01, SM2-01, dan JMB-02 semuanya sudah ditandai non-VALID oleh alat. Namun `qc_flag` (a) tidak menandai satuan salah label (ppm), (b) menandai BJB-01 100% `VALID` padahal polanya menyimpang, dan (c) memberi `SUSPECT` pada nilai saturasi SM2-01 yang kemungkinan merupakan episode asap nyata. Catatan: angka `999` pada CO (mis. KTG-01 dan KTG-02, berstatus `VALID`) **bukan** sentinel karena 999 µg/m³ adalah konsentrasi CO yang wajar.
7. **Nilai ekstrem yang mungkin nyata.** PM2.5 maksimum: KTG-01 1.620, KTG-02 1.531, PKN-01 646 µg/m³. Dari 2.009 baris PM >500 µg/m³ di tiga stasiun itu, 1.939 berstatus `VALID`. Ketiganya berada di kawasan gambut/hutan, jadi nilai ini **tidak boleh dibuang otomatis** sebagai pencilan; verifikasi dengan CO, rasio PM2.5/PM10, dan musim pada EDA.
8. **Anomali yang perlu dijelaskan sebelum analisis hubungan PM2.5–kesehatan.** (a) BJB-01 berpola musiman menyimpang dari stasiun gambut lain (r = 0.26 vs 0.86–0.98; puncak bulan ke-12, bukan Sep–Okt) dan 100% `VALID` meski unit ppm. (b) BJB-01 dan SM2-01 berada di peringkat PM2.5 ke-1 dan ke-2, tetapi kasus pernapasan RS pasangannya di peringkat ke-25 dan ke-24 (dari 27). Penjelasan yang mungkin (belum diuji): paparan di stasiun tidak mewakili populasi RS pasangannya, atau hubungan PM2.5–kasus pada data sintetis ini memang lemah di wilayah tersebut. **Tidak boleh disimpulkan sebagai kausalitas.**

## 4. Temuan data kesehatan
- 27.491 baris, 27 RS, 2023-12-31 s.d. 2026-10-14. **Tidak ada duplikat** (tanggal, RS), **tidak ada missing/teks/negatif** pada kolom kasus maupun cuaca, `dow` cocok dengan tanggal, dan kewajaran fisik terpenuhi (titik embun ≤ suhu, RH 0–100).
- 22 kombinasi RS×hari tidak ada, **seluruhnya 2023-12-31**: hanya 5 RS yang punya baris 2023-12-31 (sebelum AQMS mulai). **Dalam periode AQMS tidak ada hari hilang** (0).
- `time` adalah nomor urut hari **per RS** (bukan indeks global): 0 ketidakcocokan terhadap urutan per RS.
- Semua kolom kasus berupa bilangan bulat dan **over-dispersed** (varians/rata-rata 1.3–11.9), sehingga model hubungan harus memakai Poisson dengan dispersi (quasi-Poisson/negatif binomial), bukan OLS.
- `resp` lebih kecil dari balita + lansia pada 92 baris, jadi `resp` jangan diasumsikan mencakup kolom umur. `kapasitas_tempat_tidur` terisi untuk semua 27 RS (booklet menyebut dapat kosong), tetapi 11 RS bernilai seragam 180 (kemungkinan nilai isian).
- Kasus harian per RS tidak dapat dibakukan terhadap penduduk (tidak ada denominator populasi).

## 5. Ringkasan daftar masalah (`reports/data_quality_issues.csv`)
- Total **75 masalah**: 15 Kritis, 15 Tinggi, 38 Sedang, 7 Rendah.
- Stasiun dengan masalah Kritis: JMB-02, MDN-01, MER-01, PLB-01. Hanya Tinggi: BJB-01, PSW-01, SM2-01.
- Stasiun tanpa tanda kecurigaan pada pemeriksaan awal (17): BGL-01, BGR-01, BGR-02, BTM-01, IDP-01, JKT-01, JKT-02, JKT-03, JMB-01, KTB-01, MLG-01, MRS-01, PKU-01, PLU-01, SRG-01, STG-01, YKY-01. Pada stasiun-stasiun ini baris non-`VALID` berkisar 0.5–4.0% (label `SUSPECT`/`CALIBRATING`), semuanya berunit µg/m³ dan sebagian memiliki timestamp bergeser.

## 6. Usulan aturan pembersihan untuk tahap berikutnya (belum dijalankan)
| No | Aturan | Baris/stasiun terdampak |
|---|---|---|
| 1 | Konversi `*_ppm` → µg/m³ (PM ×1000; gas ppm×BM/24,45×1000) dan beri nama kolom seragam | BJB-01, MDN-01, PSW-01 |
| 2 | Ubah teks galat, string kosong, dan sentinel (kecuali CO=999) menjadi `NaN` | 8.997 baris berteks galat; 27.876 baris bernilai mustahil |
| 3 | Terapkan batas fisik (PM ≥ 0; suhu 5–45 °C; RH 0–100; angin ≤ 40 m/s; arah 0–360°) di luar `qc_flag` | pengaman tambahan; 0 baris `VALID` bernilai mustahil pada pemeriksaan ini |
| 4 | Timestamp: agregasi ke jam penuh (floor) untuk stasiun beresolusi sub-jam; bulatkan ke jam terdekat untuk offset konstan; buang timestamp luar periode | 18 stasiun bergeser; PLB-01 sub-jam; JMB-02 1970 |
| 5 | Reindex ke 24.432 jam; jam tanpa data tetap `NaN` (jangan diimputasi untuk PM) | 3.856 jam hilang (JMB-02, MDN-01) |
| 6 | Pemetaan `qc_flag`: VALID dipakai; `SUSPECT` dikecualikan dari analisis utama **kecuali** saturasi saat asap (SM2-01) diperlakukan khusus; uji sensitivitas | 12 label |
| 7 | Kecualikan JMB-02 dari analisis konsentrasi/kesehatan; MER-01 hanya dengan peringatan kelengkapan | 2 stasiun |
| 8 | Sampel kesehatan dibatasi pada 2024-01-01…2026-10-14 | 5 baris |

## 7. Keterbatasan dan hal yang belum diverifikasi
- Faktor ppm → µg/m³ diuji dengan orde besaran terhadap `baseline_pm25` dan median stasiun lain, bukan terhadap pengukuran referensi; PM dalam "ppm" hanyalah hipotesis (mg/m³ yang salah label).
- Kriteria "mencurigakan" (rasio 0,5–2× `baseline_pm25`, PM > 500 µg/m³, selisih peringkat ≥ 10) adalah ambang praktis, bukan standar.
- PM2.5 harian pada pemeriksaan silang adalah pendekatan profiling (≥12 jam VALID/hari), **bukan** data bersih final. Korelasi musiman memakai 7 stasiun gambut (JMB-02 dikecualikan) sehingga bersifat deskriptif.
- Tidak ada pengujian kausal; semua perbandingan peringkat/korelasi hanyalah penanda untuk investigasi pada notebook 03–04.


# 02 — Data Understanding: Temuan
> Dihasilkan otomatis oleh `notebooks/02_data_understanding.ipynb`. Semua angka berasal dari `data/processed/profiling/*.parquet`,
> `reports/data_quality_issues.csv`, dan `reports/data_dictionary.csv`. **Tidak ada data mentah yang diubah atau dibersihkan** pada tahap ini.

## 1. Cakupan dan struktur data
- **33 file** dibaca: 27 file AQMS (667.981 baris), 1 file kesehatan (27.491 baris), 3 tabel master, 2 metadata.
- Periode AQMS: 2024-01-01 s.d. 2026-10-14 = **24.432 jam** per stasiun. Hanya **25 dari 27** file memuat tepat jumlah baris itu.
- Kunci antar-tabel **cocok sepenuhnya**: 24 pemeriksaan kunci/konsistensi OK. Satu-satunya ketidakcocokan adalah jumlah baris file (R27: MDN-01=20800; PLB-01=36381), ditambah 2 butir informasi (stasiun ppm, status sensor). Pasangan stasiun ↔ sensor ↔ lokasi ↔ RS (`SENS-0NN ↔ RS-XXX-NN`) lengkap 27/27 dan provinsi/kota/kecamatan RS sama dengan stasiun pasangannya (pencocokan longgar).
- Sebaran stasiun: Sumatera 10, Jawa 7, Kalimantan 6, Sulawesi 2, Papua 2; kelompok lokasi: Perkotaan 7, Jalan Raya 2, Industri 4, Gambut 8, Latar Bersih 6.
- Status sensor: 23 Active, 3 Intermittent (PLB-01, MER-01, MDN-01), 1 Defective (JMB-02). Status ini sejalan dengan stasiun terparah yang terukur (JMB-02, MER-01, MDN-01, PLB-01); SM2-01 berstatus Active meski hanya 85.0% VALID.

## 2. Hasil verifikasi masalah yang sudah diketahui
| klaim                                                                    | bukti_terukur                                                                                                                                                                                                                                                                                 | status                                                                                                 |
|:-------------------------------------------------------------------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|:-------------------------------------------------------------------------------------------------------|
| BJB-01, MDN-01, PSW-01 memakai kolom *_ppm                               | Stasiun berkolom *_ppm: BJB-01, MDN-01, PSW-01; sites.pm25_unit konsisten (cek R16/R17)                                                                                                                                                                                                       | Terkonfirmasi                                                                                          |
| Konversi PM ×1000 dan gas ppm×BM/24,45×1000                              | 18/18 kombinasi stasiun×polutan masuk rentang kewajaran orde besaran terhadap acuan                                                                                                                                                                                                           | Didukung (uji orde besaran)                                                                            |
| JMB-02: teks error, -9999, timestamp 1970, ~0,5% VALID                   | 12 jenis teks galat (termasuk 0xDEADBEEF, ERR_COMM_TIMEOUT, #VALUE!); 5.535 sel bernilai -9999; timestamp tertua 1970-01-01 (132 baris di luar periode); VALID 0.50%                                                                                                                          | Terkonfirmasi                                                                                          |
| PLB-01: ~36 rb baris (duplikat per jam)                                  | 36.381 baris; 11.949 baris lebih pada 3.986 jam; 15.933 baris berjendela 15 menit dalam 3 blok: 01 Sep 2024–31 Oct 2024, 01 Sep 2025–31 Oct 2025, 01 Sep 2026–14 Oct 2026; 3.956 jam berisi PM2.5 berbeda-beda                                                                                | Terkonfirmasi, dengan koreksi penyebab: bacaan sub-jam (:15/:30/:45) pada Sep–Okt, bukan salinan baris |
| MDN-01: ~3.600 jam hilang                                                | 3.632 jam hilang (14.9%); 20.800 baris; gap terpanjang 121 jam                                                                                                                                                                                                                                | Terkonfirmasi                                                                                          |
| MER-01: ~20% VALID, PM negatif, suhu 99                                  | VALID 20.02%; 1.780 baris PM2.5 negatif; 749 baris suhu di luar 5–45 °C (maks 99 °C); semua baris mustahil berstatus SUSPECT (VALID-tapi-mustahil = 0)                                                                                                                                        | Terkonfirmasi                                                                                          |
| SM2-01: nilai 999 = saturasi alat (bertanda SUSPECT)                     | 2.078 sel bernilai 999 pada kolom pm10_ug_m3, pm25_ug_m3; status qc: SUSPECT; baris 999 per bulan kalender: Jul (285), Agu (267), Sep (270), Okt (217) (musiman, hanya pada musim kemarau)                                                                                                    | Terkonfirmasi                                                                                          |
| Timestamp bergeser (mis. 23:48, 00:11, milidetik)                        | 18 dari 27 stasiun punya timestamp bergeser; geseran bertanda terkecil -15.00 mnt, terbesar 30.00 mnt; offset konstan terbesar: BTM-01 -11.75 mnt, SRG-01 +11.38 mnt, JMB-02 +5.20 mnt; 18 dari 27 file memakai format dengan pecahan detik (.000); 3.983 baris tepat :30 (pembulatan ambigu) | Terkonfirmasi (geseran positif dan negatif; format milidetik)                                          |
| qc_flag: VALID, SUSPECT, CALIBRATING, WARMUP, MAINTENANCE, MISSING, dll. | 12 label ditemukan: CALIBRATING, COMM_LOST, CORRUPT, FATAL_ERR, GARBAGE, MAINTENANCE, MISSING, PARSER_FAIL, SENSOR_DEAD, SUSPECT, VALID, WARMUP                                                                                                                                               | Terkonfirmasi (+ label galat alat di JMB-02)                                                           |
| BJB-01: pola musim terbalik vs stasiun gambut lain                       | Korelasi profil musiman BJB-01 = 0.26 (stasiun gambut lain 0.86–0.98); puncak BJB-01 pada bulan ke-12 vs bulan 9, 10 pada stasiun lain                                                                                                                                                        | Terkonfirmasi (menyimpang kuat)                                                                        |
| BJB-01 & SM2-01: PM2.5 tinggi tapi kasus RS rendah                       | BJB-01: peringkat PM2.5 1 vs peringkat kasus `resp` 25 (resp/100 TT = 3.5); SM2-01: peringkat PM2.5 2 vs peringkat kasus `resp` 24 (resp/100 TT = 2.3)                                                                                                                                        | Terkonfirmasi                                                                                          |
| Relasi RS-XXX-NN ↔ SENS-0NN ↔ LOC di sites.csv                           | R20–R23: R09=OK, R20=OK, R21=OK, R22=OK, R23=OK                                                                                                                                                                                                                                               | Terkonfirmasi                                                                                          |
| 27 stasiun, 27 RS, 24.432 jam, 2024-01-01…2026-10-14                     | 27 stasiun; 27 RS; stasiun dengan tepat 24.432 baris: 25/27                                                                                                                                                                                                                                   | Terkonfirmasi                                                                                          |

## 3. Temuan utama kualitas data AQMS
1. **Satuan tidak seragam.** BJB-01, MDN-01, PSW-01 memakai kolom `*_ppm` untuk **semua** polutan, termasuk PM. Median PM2.5 setelah ×1000 berada pada 43.5–46.9 µg/m³, sangat dekat dengan `baseline_pm25` stasiunnya (35.0–54.3 µg/m³); tanpa konversi, nilainya ±1000× lebih kecil. Hipotesis konversi didukung pada 18/18 kombinasi (uji orde besaran).
2. **Satu stasiun tidak layak pakai: JMB-02 (Muaro Jambi).** Hanya 0.50% baris `VALID`; 8.997 baris (36.8%) memuat teks galat dari 7 kelompok jenis (galat perangkat keras sensor (15.988 sel), galat komunikasi/integritas data (10.670 sel), kode heksadesimal (isi memori) (5.706 sel), galat formula spreadsheet (5.042 sel), nilai tak hingga (5.032 sel), byte kosong/null (460 sel), kode galat umum (273 sel)); 24.308 baris (99.5%) memuat nilai mustahil; 132 baris bertimestamp 1970; PM2.5 > PM10 pada 4.897 baris. Status sensor `Defective`. Perlakuan awal: kecualikan dari analisis konsentrasi dan kesehatan, tetapi **pertahankan sebagai bukti** untuk Tujuan 5 (kecukupan jaringan).
3. **Stasiun dengan cakupan VALID rendah.** MER-01: 20.0% VALID (9.778 baris berisi sel kosong, 2.529 baris mustahil, semuanya `SUSPECT`). MDN-01: 3.632 jam tidak ada baris (14.9%), gap terpanjang 121 jam, jam VALID hanya 68.1% dari 24.432. SM2-01: 85.0% VALID; 1.039 baris bernilai 999 pada PM2.5/PM10, seluruhnya `SUSPECT` dan hanya pada bulan Jul, Agu, Sep, Okt (musim kemarau), konsisten dengan dugaan saturasi alat saat asap pekat. Pengecualian `SUSPECT` secara membabi buta berisiko **menghapus episode asap puncak** pada SM2-01.
4. **PLB-01: baris berlebih berasal dari logger yang merekam tiap 15 menit pada musim asap.** 36.381 baris vs 24.432 jam. 15.933 baris berjendela 15 menit, terkumpul dalam 3 blok jam berurutan (01 Sep 2024–31 Oct 2024; 01 Sep 2025–31 Oct 2025; 01 Sep 2026–14 Oct 2026), yaitu **hanya pada September–Oktober** (4 bacaan/jam pada :00/:15/:30/:45); di luar blok itu data per jam normal. Ini bukan duplikat murni: aturan "hapus duplikat" akan membuang data sah, sedangkan pembulatan ke jam terdekat memindahkan baris :45 ke jam berikutnya dan menciptakan duplikat semu tepat pada periode asap (3.986 jam berisi >1 baris, 3.956 jam di antaranya bernilai PM2.5 berbeda). Gunakan agregasi per jam penuh (floor). Selain itu terdapat 5.136 baris `MISSING` (kosong).
5. **Timestamp.** 18 dari 27 stasiun memiliki timestamp bergeser dari awal jam (geseran bertanda -15.0 s.d. +30.0 menit; 17 stasiun memakai satu offset konstan, sisanya bervariasi). 3.983 baris tepat :30 sehingga pembulatan ke "jam terdekat" ambigu (terjadi pada PLB-01 yang sub-jam). Seluruh `timestamp_start` dapat di-parse (0 baris tak terbaca); satu-satunya timestamp di luar periode adalah 1970 pada JMB-02.
6. **`qc_flag` konsisten dengan nilai mustahil, tetapi tidak cukup sebagai satu-satunya filter.** Ditemukan 12 label (metadata hanya mendefinisikan 4). Menurut aturan rentang fisik, ada 0 baris `VALID` bernilai mustahil; nilai mustahil di MER-01, SM2-01, dan JMB-02 semuanya sudah ditandai non-VALID oleh alat. Namun `qc_flag` (a) tidak menandai satuan salah label (ppm), (b) menandai BJB-01 100% `VALID` padahal polanya menyimpang, dan (c) memberi `SUSPECT` pada nilai saturasi SM2-01 yang kemungkinan merupakan episode asap nyata. Catatan: angka `999` pada CO (mis. KTG-01 dan KTG-02, berstatus `VALID`) **bukan** sentinel karena 999 µg/m³ adalah konsentrasi CO yang wajar.
7. **Nilai ekstrem yang mungkin nyata.** PM2.5 maksimum: KTG-01 1.620, KTG-02 1.531, PKN-01 646 µg/m³. Dari 2.009 baris PM >500 µg/m³ di tiga stasiun itu, 1.939 berstatus `VALID`. Ketiganya berada di kawasan gambut/hutan, jadi nilai ini **tidak boleh dibuang otomatis** sebagai pencilan; verifikasi dengan CO, rasio PM2.5/PM10, dan musim pada EDA.
8. **Anomali yang perlu dijelaskan sebelum analisis hubungan PM2.5–kesehatan.** (a) BJB-01 berpola musiman menyimpang dari stasiun gambut lain (r = 0.26 vs 0.86–0.98; puncak bulan ke-12, bukan Sep–Okt) dan 100% `VALID` meski unit ppm. (b) BJB-01 dan SM2-01 berada di peringkat PM2.5 ke-1 dan ke-2, tetapi kasus pernapasan RS pasangannya di peringkat ke-25 dan ke-24 (dari 27). Penjelasan yang mungkin (belum diuji): paparan di stasiun tidak mewakili populasi RS pasangannya, atau hubungan PM2.5–kasus pada data sintetis ini memang lemah di wilayah tersebut. **Tidak boleh disimpulkan sebagai kausalitas.**

## 4. Temuan data kesehatan
- 27.491 baris, 27 RS, 2023-12-31 s.d. 2026-10-14. **Tidak ada duplikat** (tanggal, RS), **tidak ada missing/teks/negatif** pada kolom kasus maupun cuaca, `dow` cocok dengan tanggal, dan kewajaran fisik terpenuhi (titik embun ≤ suhu, RH 0–100).
- 22 kombinasi RS×hari tidak ada, **seluruhnya 2023-12-31**: hanya 5 RS yang punya baris 2023-12-31 (sebelum AQMS mulai). **Dalam periode AQMS tidak ada hari hilang** (0).
- `time` adalah nomor urut hari **per RS** (bukan indeks global): 0 ketidakcocokan terhadap urutan per RS.
- Semua kolom kasus berupa bilangan bulat dan **over-dispersed** (varians/rata-rata 1.3–11.9), sehingga model hubungan harus memakai Poisson dengan dispersi (quasi-Poisson/negatif binomial), bukan OLS.
- `resp` lebih kecil dari balita + lansia pada 92 baris, jadi `resp` jangan diasumsikan mencakup kolom umur. `kapasitas_tempat_tidur` terisi untuk semua 27 RS (booklet menyebut dapat kosong), tetapi 11 RS bernilai seragam 180 (kemungkinan nilai isian).
- Kasus harian per RS tidak dapat dibakukan terhadap penduduk (tidak ada denominator populasi).

## 5. Ringkasan daftar masalah (`reports/data_quality_issues.csv`)
- Total **75 masalah**: 15 Kritis, 15 Tinggi, 38 Sedang, 7 Rendah.
- Stasiun dengan masalah Kritis: JMB-02, MDN-01, MER-01, PLB-01. Hanya Tinggi: BJB-01, PSW-01, SM2-01.
- Stasiun tanpa tanda kecurigaan pada pemeriksaan awal (17): BGL-01, BGR-01, BGR-02, BTM-01, IDP-01, JKT-01, JKT-02, JKT-03, JMB-01, KTB-01, MLG-01, MRS-01, PKU-01, PLU-01, SRG-01, STG-01, YKY-01. Pada stasiun-stasiun ini baris non-`VALID` berkisar 0.5–4.0% (label `SUSPECT`/`CALIBRATING`), semuanya berunit µg/m³ dan sebagian memiliki timestamp bergeser.

## 6. Usulan aturan pembersihan untuk tahap berikutnya (belum dijalankan)
| No | Aturan | Baris/stasiun terdampak |
|---|---|---|
| 1 | Konversi `*_ppm` → µg/m³ (PM ×1000; gas ppm×BM/24,45×1000) dan beri nama kolom seragam | BJB-01, MDN-01, PSW-01 |
| 2 | Ubah teks galat, string kosong, dan sentinel (kecuali CO=999) menjadi `NaN` | 8.997 baris berteks galat; 27.876 baris bernilai mustahil |
| 3 | Terapkan batas fisik (PM ≥ 0; suhu 5–45 °C; RH 0–100; angin ≤ 40 m/s; arah 0–360°) di luar `qc_flag` | pengaman tambahan; 0 baris `VALID` bernilai mustahil pada pemeriksaan ini |
| 4 | Timestamp: agregasi ke jam penuh (floor) untuk stasiun beresolusi sub-jam; bulatkan ke jam terdekat untuk offset konstan; buang timestamp luar periode | 18 stasiun bergeser; PLB-01 sub-jam; JMB-02 1970 |
| 5 | Reindex ke 24.432 jam; jam tanpa data tetap `NaN` (jangan diimputasi untuk PM) | 3.856 jam hilang (JMB-02, MDN-01) |
| 6 | Pemetaan `qc_flag`: VALID dipakai; `SUSPECT` dikecualikan dari analisis utama **kecuali** saturasi saat asap (SM2-01) diperlakukan khusus; uji sensitivitas | 12 label |
| 7 | Kecualikan JMB-02 dari analisis konsentrasi/kesehatan; MER-01 hanya dengan peringatan kelengkapan | 2 stasiun |
| 8 | Sampel kesehatan dibatasi pada 2024-01-01…2026-10-14 | 5 baris |

## 7. Keterbatasan dan hal yang belum diverifikasi
- Faktor ppm → µg/m³ diuji dengan orde besaran terhadap `baseline_pm25` dan median stasiun lain, bukan terhadap pengukuran referensi; PM dalam "ppm" hanyalah hipotesis (mg/m³ yang salah label).
- Kriteria "mencurigakan" (rasio 0,5–2× `baseline_pm25`, PM > 500 µg/m³, selisih peringkat ≥ 10) adalah ambang praktis, bukan standar.
- PM2.5 harian pada pemeriksaan silang adalah pendekatan profiling (≥12 jam VALID/hari), **bukan** data bersih final. Korelasi musiman memakai 7 stasiun gambut (JMB-02 dikecualikan) sehingga bersifat deskriptif.
- Tidak ada pengujian kausal; semua perbandingan peringkat/korelasi hanyalah penanda untuk investigasi pada notebook 03–04.
